# Devoir maison 1 — carnets J1 et J2

> **À rendre avant la première séance de la semaine suivante.** Ce devoir porte sur les carnets
> **J1** (du fichier brut à la soumission) et **J2** (ce que la table cache). Il se fait **seul**, à
> la maison, et il se rend.

**Ce qu'on rend :** ce carnet, rempli, déposé dans votre **dépôt GitHub personnel** avec la
déclaration d'usage de l'IA. **Les trois tableaux** de la fin et **l'outil de la question 12** sont
les pièces principales.

**Pourquoi ce devoir :** chaque question demande une mesure que **vous** produisez — jamais une
lecture. Presque toutes s'ouvrent sur **une prédiction écrite avant la mesure** : c'est le geste du
métier, et c'est aussi ce qui rend une erreur visible.

**Quatre mots, pour être sûr :**

- **les deux réponses** — le patient vient (`patient_absent` vaut 0), ou il ne vient pas (1) ;
- **la référence** — le score d'un modèle qui répond toujours la même chose : le minimum à battre ;
- **la barre de bruit** — de combien un score bouge tout seul quand seul le découpage change ;
  en dessous de **deux barres**, un écart n'est pas une différence ;
- **un constat** — une observation, un chiffre, **par rapport à quoi**, et ce que vous en faites.

**Quinze questions, dont douze de cœur.** Chaque titre porte sa mention : **cœur** = le socle du
semestre ; **approfondissement** = à faire dès que le cœur tient. Si le temps manque, faites les
douze de cœur d'abord — les trois approfondissements portent les gestes les plus fins, et ils se
reprennent plus tard. Rien n'est facultatif au sens où tout est lu.

---

## Comment ça marche

1. La partie **« Mise en place »** est **fournie** : exécutez-la de haut en bas. **Deux de ses
   cellules sont faites pour être touchées** : votre **identifiant GitHub** (les questions 2 et 11
   mesurent sur un échantillon tiré de lui), et `ajouter_features` (la question 10).
   **Un identifiant mal recopié change votre échantillon** : copiez-le comme dans l'URL de votre dépôt.
2. Chaque **question** porte sa consigne, en markdown, un **emplacement vide**, et des
   **contrôles de forme** posés dans la cellule : ils disent si votre réponse a la forme attendue
   (un pourcentage, une dimension, une cohérence avec **vos** autres réponses). Ils ne donnent
   **jamais** la valeur attendue, et ils se taisent tant que vous n'avez rien écrit.
3. Les questions sont dans l'ordre des carnets : **1 à 8 sur J1**, **9 à 15 sur J2** — sauf la 15,
   qui transpose la méthode hors de ce jeu de données.
4. Une réponse se **chiffre** ou s'**écrit**. Plusieurs questions demandent d'**annoncer AVANT de
   mesurer** : `noter(...)` refuse d'ailleurs d'enregistrer un essai non annoncé.
5. À la fin : les **trois tableaux**, puis le **dépôt**.

**Ce qui est lu.** **Les trois tableaux** — constats, hypothèses, essais — **et le code**, là où il
produit un chiffre. **Ce dépôt est relevé** : il est lu, et **les sept devoirs comptent ensemble
pour 10 % de la note**, dans la composante « exercices et devoirs » (25 %, avec E1, E2, E3).
**Il n'y a pas d'oral sur les devoirs** : l'oral individuel, en fin de parcours, porte sur
l'ensemble de votre travail — les trois blocs et les deux terrains.

> **La règle, la même qu'en séance :** un score sans référence ne veut rien dire, un écart plus
> petit que deux barres n'est pas une différence, et un constat sans chiffre n'en est pas un.


## Mise en place (fournie)

Ces cellules sont **extraites de `notebook_J1.ipynb` et de `notebook_J2.ipynb`**, qui restent les
sources uniques de la table de travail, de la référence et du modèle. Exécutez-les de haut en bas,
sans rien y modifier — sauf `ajouter_features`, que la question 10 vous demandera d'écrire.

Les `random_state=0` que vous verrez sont la **graine du tirage au sort** (expliquée en J1 §5) :
le même nombre redonne le même découpage, donc la même mesure.


### Les outils


In [ ]:
# les outils
# Aucun import en bloc au début : chaque module entre dans le carnet au moment de son premier
# usage. On voit ainsi à quoi il sert, au lieu de le deviner dans une liste.

# `display` : un tableau s'affiche proprement dans un carnet, et reste lisible ailleurs.
# C'est par `display` qu'on montre TOUTES les tables de ce carnet — jamais un print de texte.
try:
    from IPython.display import display
except ImportError:
    def display(x):
        print(x)


### Les données


In [ ]:
import os
import pandas as pd
# où sont les fichiers — on CHERCHE `train.csv`, on ne suppose pas un chemin
# (Kaggle a déjà déplacé ses points de montage : un chemin recopié se périme, une recherche non)
def _trouver_dossier(racines):
    """Première racine contenant train.csv — en surface, puis en profondeur."""
    for racine in racines:
        if not os.path.isdir(racine):
            continue
        if os.path.exists(os.path.join(racine, "train.csv")):
            return racine
        for chemin, _, fichiers in os.walk(racine):
            if "train.csv" in fichiers:
                return chemin
    return None

DOSSIER = _trouver_dossier(["/kaggle/input", "sortie"])
assert DOSSIER, "train.csv introuvable : les données de la compétition sont-elles montées ?"


In [ ]:
train = pd.read_csv(os.path.join(DOSSIER, "train.csv"), low_memory=False)
test = pd.read_csv(os.path.join(DOSSIER, "test.csv"), low_memory=False)

ID, CIBLE = "ligne_id", "patient_absent"


### La table de travail


In [ ]:
A_ECARTER = [ID, "dossier_id", "patient_id", "duree_consultation", "constantes_relevees",
             "motif_absence", "date_prise", "date_rdv", "taille_cm", "date_naissance",
             "canal_prise"]

def matrice(cols):
    """La table de travail : colonnes écartées retirées, catégorielles déclarées."""
    X = cols.drop(columns=[c for c in A_ECARTER + [CIBLE] if c in cols.columns]).copy()
    cat = [c for c in X.columns if not pd.api.types.is_numeric_dtype(X[c])]
    trop_fin = [c for c in cat if X[c].nunique() > 255]
    X = X.drop(columns=trop_fin)
    cat = [c for c in cat if c not in trop_fin]
    for c in cat:
        X[c] = X[c].fillna("").astype("category")
    return X, cat, trop_fin

X, CAT, TROP_FIN = matrice(train)
Xt, _, _ = matrice(test)

for c in CAT:
    Xt[c] = Xt[c].cat.set_categories(X[c].cat.categories)


### La cible, et la référence


In [ ]:
repartition = pd.DataFrame({"lignes": [len(train), int(train[CIBLE].sum())],
                            "part (%)": [100.0, round(100 * train[CIBLE].mean(), 2)]},
                           index=["toutes", "absents"])

reference = 100 * max(train[CIBLE].mean(), 1 - train[CIBLE].mean())


In [ ]:
y = train[CIBLE]
reference = 100 * max(y.mean(), 1 - y.mean())


### Le premier modèle


In [ ]:
# le modèle et le découpage : ils entrent ici
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.model_selection import train_test_split
Xtr, Xval, ytr, yval = train_test_split(X, y, test_size=0.2, stratify=y, random_state=0)

modele = HistGradientBoostingClassifier(random_state=0).fit(Xtr, ytr)
score_interne = 100 * modele.score(Xval, yval)


### Vos features — LA cellule que vous modifiez


In [ ]:
# votre feature : elle reçoit la TABLE DE TRAVAIL — identifiants, dates et colonnes post-issue
# déjà retirés — et la rend enrichie. `predict` l'appelle sur la table BRUTE, après avoir retiré
# les mêmes colonnes : le crochet voit donc la même forme des deux côtés, ce qui est la règle.
def ajouter_features(df):
    """Vos features maison. Appliquée identiquement à l'entraînement et au test."""
    df = df.copy()
    # Exemple — remplacez-le par la vôtre :
    # df["plus_de_60_ans"] = (df["age"] > 60).astype(int)
    return df

# Votre feature entre par `preparer()`, la chaîne unique : on n'applique RIEN à la main.
# Après l'avoir écrite, appelez `remesurer()` (au devoir, dans la cellule prévue) : `X` la
# reçoit une seule fois, et `predict` l'applique aussi, de son côté, sur la table brute.


### La chaîne — une seule, des deux côtés


In [ ]:
# ── LA CHAÎNE : une seule fonction, celle que `predict` ET l'entraînement utilisent.
def preparer(brut):
    """Écarte, type, applique vos features : LA chaîne, des deux côtés.
       `predict` l'appelle ; l'entraînement l'appelle aussi, par `matrice()` sur la table de
       travail. Contourner cette fonction, c'est apprendre sur une table et prédire sur une
       autre — exactement le défaut que la séance 2 traque."""
    d = brut.drop(columns=[c for c in A_ECARTER + TROP_FIN + [CIBLE] if c in brut.columns],
                  errors="ignore").copy()
    for c in CAT:
        if c in d.columns:
            d[c] = d[c].fillna("").astype("category").cat.set_categories(X[c].cat.categories)
    return ajouter_features(d)

def remesurer():
    """Réentraîne sur la table de travail COURANTE et met `modele` à jour.
       `ajouter_features` n'est appliquée qu'UNE fois, depuis X : ré-exécutable sans que la
       feature se superpose à elle-même."""
    global modele
    Xc = ajouter_features(X)
    a, b, c, d = train_test_split(Xc, y, test_size=0.2, stratify=y, random_state=0)
    modele = HistGradientBoostingClassifier(random_state=0).fit(a, c)
    return 100 * modele.score(b, d)


### L'interface — le modèle qu'on peut appeler


In [ ]:
# numpy, pour la seule assertion qui suit
import numpy as np

def predict(df):
    """Renvoie un tableau de 0/1, un par ligne de df. C'est cette fonction que le cours appellera.
       Elle passe par `preparer(...)`, la même chaîne que l'entraînement — pas une copie."""
    # elle ne refait PAS la chaîne à la main : elle appelle `preparer`, celle de l'entraînement.
    # C'est ce couplage qui garantit qu'une table n'est jamais préparée de deux façons.
    return modele.predict(preparer(df)).astype(int)

def test_predict():
    """Vérifie que l'interface répond ce qu'on attend d'elle."""
    p = predict(test.head(500))

    assert len(p) == 500, f"attendu 500 prédictions, obtenu {len(p)}"
    assert set(np.unique(p)) <= {0, 1}, "les valeurs doivent être 0 ou 1"
    print(f"test OK — 500 prédictions, {int(sum(p))} positifs")

# test_predict()   (au devoir : la cellule « l'interface » s'en charge)


### Le tableau d'essais


In [ ]:
# l'horloge entre ici : c'est elle qui rend une prédiction vérifiable
from datetime import datetime

ESSAIS = []
PREDICTIONS = []

def prevoir(essai, attendu, raison):
    """Écrit ce que vous attendez AVANT de mesurer — horodaté.
       `noter(...)` refuse ensuite tout essai sans prédiction : l'ordre n'est pas demandé, il est
       appliqué. Ce que cette fonction garantit, exactement : une prédiction EXISTAIT avant que la
       ligne du tableau soit écrite. Elle ne date pas la mesure — le score lui arrive déjà calculé.
       Pour la garantie complète, mesurez dans `essai(...)`, qui date la mesure lui-même."""
    PREDICTIONS[:] = [p for p in PREDICTIONS if p["essai"] != essai]
    PREDICTIONS.append({"essai": essai, "attendu": attendu, "raison": raison,
                        "écrit à": datetime.now().strftime("%H:%M:%S")})
    display(pd.DataFrame(PREDICTIONS))

def essai(nom, mesure, changement="", attendu=""):
    """Le chemin STRICT : la mesure est prise ICI, donc l'ordre (prédiction puis mesure) est prouvé.
       `mesure` est une fonction sans argument qui rend le score, par exemple
       `lambda: modele.score(Xval, yval)`. Refuse de mesurer sans prédiction."""
    prevu = [p for p in PREDICTIONS if p["essai"] == nom]
    if not prevu:
        print(f"refusé : aucune prédiction pour « {nom} ». Écrivez prevoir(...) d'abord.")
        return None
    score = 100 * mesure()
    noter(nom, changement or "voir la question", attendu or prevu[0]["attendu"], score)
    return score

def noter(hypothese, changement, attendu, score):
    """Ajoute une ligne au tableau. `attendu` : un signe et une raison, jamais une valeur.
       Ré-exécuter une cellule REMPLACE la ligne de même `hypothèse` : elle ne la double pas.

       REFUSE d'enregistrer un essai non annoncé — c'est la règle du cours, appliquée.
       Ce qu'elle garantit, et rien de plus : une prédiction existait AVANT que cette ligne soit
       écrite. Elle ne date pas la mesure — d'où `essai(...)` pour la garantie complète."""
    prevu = [p for p in PREDICTIONS if p["essai"] == hypothese]
    if not prevu:
        print(f"  refusé : aucune prédiction pour « {hypothese} ».")
        print("  Écrivez prevoir(...) d'abord — l'ordre est vérifié, pas seulement demandé.")
        return
    ligne = {"hypothèse": hypothese, "changement": changement,
             "attendu": attendu, "mesuré": round(score, 2),
             "prédiction écrite avant ?": "oui, à " + prevu[0]["écrit à"]}
    ESSAIS[:] = [e for e in ESSAIS if e["hypothèse"] != hypothese] + [ligne]
    display(pd.DataFrame(ESSAIS))

# les deux amorces du carnet sont annoncées elles aussi — sinon elles ne passeraient pas
prevoir("rien faire du tout", "c'est le minimum à battre",
        "un modèle qui répond toujours la même chose vaut la classe majoritaire")
prevoir("un modèle suffit-il ?", "mieux que la référence",
        "les arbres captent une partie du signal, pas tout")


### La soumission


In [ ]:
soumission = pd.DataFrame({ID: test[ID], CIBLE: modele.predict(Xt).astype(int)})
soumission.to_csv("submission.csv", index=False)


### Le modèle linéaire


In [ ]:
# le modèle linéaire : il entre ici
from sklearn.linear_model import LogisticRegression
X_num = pd.get_dummies(X, columns=CAT).fillna(0)
Xtr_n, Xval_n = X_num.loc[Xtr.index], X_num.loc[Xval.index]

modele_lineaire = LogisticRegression(solver="newton-cholesky").fit(Xtr_n, ytr)
score_lineaire = 100 * modele_lineaire.score(Xval_n, yval)


### Les outils du devoir : les tableaux, la barre, les contrôles


In [ ]:
# les trois tableaux, et les outils qui les remplissent

CONSTATS = []

def constat(observation, chiffre, comparaison, consequence):
    """observation : ce qu'on a vu · chiffre : la mesure exacte
       comparaison : par rapport à quoi ? (un nombre) · conséquence : garder, écarter,
       surveiller, mesurer. Ré-exécuter une cellule REMPLACE la ligne de même `observation`."""
    ligne = {"observation": observation, "chiffre": chiffre,
             "comparaison": comparaison, "conséquence": consequence}
    CONSTATS[:] = [c for c in CONSTATS if c["observation"] != observation] + [ligne]
    display(pd.DataFrame(CONSTATS))

HYPOTHESES = []

def hypothese(feature, ce_qu_elle_raconte, comment_on_la_test, ce_qui_la_contredirait):
    """Une hypothèse est une prédiction qu'on peut tester, avec un critère annoncé AVANT.
       Ré-exécuter une cellule REMPLACE la ligne de même `feature`."""
    ligne = {"feature": feature, "ce qu'elle raconte": ce_qu_elle_raconte,
             "test": comment_on_la_test, "contredite si": ce_qui_la_contredirait}
    HYPOTHESES[:] = [h for h in HYPOTHESES if h["feature"] != feature] + [ligne]
    display(pd.DataFrame(HYPOTHESES))

# les trois constats de la séance 1 — ils comptent : la question 3 en demande CINQ au total,
# donc DEUX des vôtres. Ils sont réécrits ici parce que ce carnet doit tourner seul.
constat("une colonne ne dit rien",
        f"{train['zone_clinique'].nunique()} valeur(s) distincte(s)",
        f"sur les {len(train)} lignes de la table, soit une colonne constante au sens du cours",
        "l'écarter — et ce qu'on y gagne se mesure")
constat("deux colonnes disent la même chose",
        "corrélation 1,000 entre cout_acte et montant_acte",
        f"contre moins de 0,05 pour les autres paires sur {len(train)} lignes : un rapport constant",
        "n'en garder qu'une")
constat("la cible est déséquilibrée",
        f"{100 * train[CIBLE].mean():.2f} % d'absents",
        "contre 50 % pour un problème équilibré : la référence monte d'autant",
        "mesurer : cela conditionne la métrique, donc ce qu'on regarde")
CONSTATS_DE_LA_SEANCE = [c['observation'] for c in CONSTATS]

# les deux amorces de J1 — annoncées, puis notées (c'est la règle de l'ordre, montrée)
noter("rien faire du tout", "classe majoritaire", "c'est le minimum à battre", reference)
noter("un modèle suffit-il ?", "HistGradientBoosting", "mieux que la référence", score_interne)

# la barre de bruit — recalculée ici parce que la question 5 en change la taille du jeu
erreur_type = 100 * ((score_interne / 100) * (1 - score_interne / 100) / len(yval)) ** 0.5
print(f"erreur-type de l'exactitude de validation (jeu de 20 %) : ± {erreur_type:.2f} point(s)")
print("  formule : racine de p(1-p)/n, où n est le nombre de lignes de validation.")
print("  vocabulaire : une barre de bruit EST une erreur-type — « deux barres » = deux erreurs-types.")

# ── VOTRE échantillon : il dépend de VOTRE identifiant GitHub, et de lui seul.
#    Les questions 2 et 11 mesurent dessus : deux voisins n'ont donc pas les mêmes chiffres,
#    et un chiffre recopié ne tombe pas juste. Le reste du devoir (le socle J1/J2) garde le
#    protocole figé du cours — c'est voulu : ce qui doit être comparable entre tous reste
#    comparable, et ce qui doit être VOTRE mesure est la vôtre.
IDENTIFIANT_GITHUB = "…"          # <- remplacez par votre pseudo GitHub, comme dans l'URL
import hashlib
if IDENTIFIANT_GITHUB.strip() and IDENTIFIANT_GITHUB.strip() != '…':
    GRAINE = int(hashlib.sha256(IDENTIFIANT_GITHUB.strip().lower().encode('utf-8')).hexdigest()[:8], 16)
else:
    GRAINE = None
mon_train = train.sample(frac=0.6, random_state=GRAINE) if GRAINE is not None else train
print(f"votre échantillon : {len(mon_train)} lignes sur {len(train)}"
      + ("" if GRAINE is not None else "   (identifiant non renseigné : la table entière)"))

# la table de travail NUE, gardée de côté : c'est sur elle que `remesurer()` rebâtit `X`,
# pour que votre feature s'applique UNE SEULE fois, même si vous relancez la cellule dix fois.
X_sans_features = matrice(train)[0]

def remesurer():
    """Réentraîne sur la table de travail COURANTE et met `modele` à jour — et `X` avec.
       `ajouter_features` n'est appliquée qu'UNE fois, depuis la table nue : ré-exécutable."""
    global modele, X
    X = ajouter_features(X_sans_features)
    a, b, c, d = train_test_split(X, y, test_size=0.2, stratify=y, random_state=0)
    modele = HistGradientBoostingClassifier(random_state=0).fit(a, c)
    return 100 * modele.score(b, d)

VERIFICATIONS = []

def verifier(nom, condition, indice=''):
    """Contrôle silencieux : il ne dit JAMAIS la valeur attendue."""
    VERIFICATIONS.append({'contrôle': nom, 'tient': bool(condition)})
    if condition:
        print('  ok        ' + nom)
    else:
        print('  À REVOIR  ' + nom + '   ->   ' + indice)

def controle(nom, quoi, indice='', des_que=None):
    """Contrôle de FORME, posé dans la cellule de la question. Il ne compare JAMAIS à une valeur
       attendue : un type, une plage, une dimension, ou une cohérence avec VOS autres réponses.
       Tant que la réponse n'est pas écrite, il se tait.

       `des_que` : une fonction sans argument qui dit si la réponse EXISTE déjà. Tant qu'elle rend
       False, le contrôle se tait — un contrôle parle d'une réponse écrite, jamais de son absence."""
    if des_que is not None:
        try:
            if not des_que():
                print(f'  (pas encore écrit)  {nom}')
                return
        except NameError:
            print(f'  (pas encore écrit)  {nom}')
            return
    try:
        ok = bool(quoi())
    except NameError:
        print(f'  (pas encore écrit)  {nom}')
        return
    except Exception as e:
        print(f'  REVOIR    {nom}   ->   {type(e).__name__} : {e}')
        return
    VERIFICATIONS.append({'contrôle': nom, 'tient': ok})
    print(('  ok        ' if ok else '  REVOIR    ') + nom + ('' if ok else '   ->   ' + indice))

# le nombre de colonnes de la table AVANT votre feature (la question 10 s'y compare)
n_colonnes_avant = X.shape[1]

# les deux essais du haut du carnet J1 sont des AMORCES : ils ne comptent pas comme les vôtres.
AMORCES = ('rien faire du tout', 'un modèle suffit-il ?')

# la clé de la question 15 vit côté enseignant, pas ici : un vocabulaire de quatre familles,
# c'est 4^6 combinaisons, et une empreinte de nom de colonne se casse en quelques secondes.


### L'interface, telle qu'on l'appelle


In [ ]:
# `predict` est définie par la mise en place, mais elle est testée ICI : ainsi, `Run All`
# fonctionne même après que vous avez écrit votre feature (la Q9 de l'ancien devoir cassait là).
# Relancez CETTE cellule après avoir touché `ajouter_features` : `X` reçoit la feature une seule
# fois, `modele` est mis à jour, et `predict` l'applique aussi de son côté.
print(f"exactitude de validation, table de travail courante : {remesurer():.2f} %")
test_predict()

import matplotlib.pyplot as plt          # pour les graphiques des questions 2 et 13


## Les questions

**Quinze questions : douze de cœur, trois d'approfondissement.** Les huit premières portent sur
**J1**, les sept dernières sur **J2**. Pour chacune : lisez la consigne, écrivez votre code (ou
votre phrase) à l'emplacement prévu, et **exécutez la cellule** — c'est l'exécution qui déclenche
les contrôles.


### Question 1 — La référence, recalculée et prouvée — et le collègue qui annonce 83 %  ·  *carnet J1*  ·  *cœur*


La référence est le chiffre qui décide si tout ce qui suit vaut quelque chose : le score
d'un modèle qui répond **toujours la même chose**. Le carnet l'a calculée sur **tout** `train`.

À faire :
  1. calculez la part d'absents **sur les lignes de contrôle** (`yval`), en **une** ligne, sans
     réutiliser `reference` → `part_absents_controle` ;
  2. **deux chiffres différents, et c'est le piège de la séance** : cette part d'absents n'est **pas**
     la référence. La référence est une **exactitude** — la part de lignes bien classées par un
     modèle qui répond toujours la classe majoritaire (ici : « le patient vient ») : calculez-la
     **sur les lignes de contrôle** → `ma_reference` ;
  3. construisez ce prédicteur : un **tableau de 0/1 de la même longueur que `yval`**, rempli de la
     classe majoritaire — puis montrez qu'il **atteint** `ma_reference` — pas « environ » ;
  4. écrivez l'écart entre `ma_reference` et `reference` (celle de la mise en place, calculée sur
     tout `train`) → `ecart_a_train`, et **pourquoi ce petit écart compte** quand on compare deux
     modèles ;
  5. **un collègue annonce 83 %** d'exactitude. Écrivez votre réponse en **deux chiffres** :
     l'écart à la référence, puis **le nombre de barres de bruit** que vaut cet écart. Un
     score qui ne bat pas la référence de deux barres n'est pas un résultat.

À écrire : `part_absents_controle`, `ma_reference`, `ecart_a_train`, `reponse_au_collegue`,
`ecart_en_barres`.


In [ ]:
# TODO : la part d'absents sur les lignes de contrôle, en UNE ligne — en pourcentage.
# part_absents_controle = … (à vous)
# TODO : la RÉFÉRENCE sur les lignes de contrôle. Attention : ce n'est PAS la même chose.
# ma_reference = … (à vous : en pourcentage)
# TODO : le prédicteur (un 0/1 par ligne de `yval`), l'écart à `reference`, puis la réponse.
# pred = … (à vous)
# ecart_a_train = … (à vous)
# reponse_au_collegue = … (à vous : deux chiffres, et ce qu'ils veulent dire)
# ecart_en_barres = … (à vous : l'écart du collègue divisé par la barre du cours)
import numpy as np
import pandas as pd

# Charger les données
DOSSIER = "/kaggle/input/competitions/rappels-et-sur-réservation-des-créneaux"

train = pd.read_csv(f"{DOSSIER}/train.csv", low_memory=False)
test = pd.read_csv(f"{DOSSIER}/test.csv", low_memory=False)

# Préparer les lignes de contrôle
# yval doit déjà contenir les vraies étiquettes des lignes de contrôle.
# Si yval existe déjà dans ton notebook, conserve-le.

# 1. Part des absents sur les lignes de contrôle, en pourcentage
part_absents_controle = (np.asarray(yval) == 1).mean() * 100

# 2. Référence : pourcentage de la classe majoritaire
ma_reference = 100 - part_absents_controle

# 3. Prédicteur : prédire toujours la classe majoritaire (0)
pred = np.zeros(len(yval), dtype=int)

# 4. Score du prédicteur, en pourcentage
score_pred = (pred == np.asarray(yval)).mean() * 100

# Écart entre le score et la référence, en points de pourcentage
ecart_a_train = score_pred - ma_reference

# 5. Réponse au collègue : deux chiffres et leur signification
reponse_au_collegue = (
    f"Part des absents : {part_absents_controle:.2f} %. "
    f"Référence majoritaire : {ma_reference:.2f} %. "
    f"Le prédicteur obtient {score_pred:.2f} %, "
    f"soit un écart de {ecart_a_train:.2f} point(s)."
)

# 6. Écart en barres du cours
# Renseigne ici la valeur réelle de la barre du cours.
barre_du_cours = None

if barre_du_cours is not None and barre_du_cours != 0:
    ecart_en_barres = ecart_a_train / barre_du_cours
    print(f"Écart en barres du cours : {ecart_en_barres:.2f}")
else:
    ecart_en_barres = None
    print("Écart en barres : indique d'abord la valeur de la barre du cours.")

# Afficher les résultats
print("\n===== RÉSULTATS =====")
print(f"Part des absents : {part_absents_controle:.2f} %")
print(f"Référence majoritaire : {ma_reference:.2f} %")
print(f"Score du prédicteur : {score_pred:.2f} %")
print(f"Écart à la référence : {ecart_a_train:.2f} points")
print("\nRéponse au collègue :")
print(reponse_au_collegue)
controle('la part d absents est un pourcentage, et c est la classe MINORITAIRE',
         lambda: 0 < part_absents_controle < 50,
         'la classe minoritaire : c est elle qu on cherche à détecter')
controle('`ma_reference` est un pourcentage, au-dessus de 50',
         lambda: 50 < ma_reference <= 100,
         'la classe majoritaire fait plus de la moitié : la référence est son score')
controle('les deux chiffres sont complémentaires',
         lambda: abs((part_absents_controle + ma_reference) - 100) < 0.05,
         'part d absents + référence = 100 % : deux faces du même tirage')
controle('le prédicteur ne répond que des 0 et des 1', lambda: set(pred) <= {0, 1})
controle('son score COÏNCIDE avec votre référence',
         lambda: abs((pred == yval).mean() * 100 - ma_reference) < 0.05,
         'les deux mesures doivent tomber ensemble : c est le point de la question')
controle('l écart à `reference` est chiffré', lambda: abs(ecart_a_train) < 5,
         'deux chiffres proches, séparés par un écart de tirage')
controle('la réponse au collègue porte deux chiffres',
         lambda: len(reponse_au_collegue) > 60 and 0 < abs(ecart_en_barres) < 30,
         'écrivez l écart à la référence ET le nombre de barres')


### Question 2 — Le balayage, le piège des valeurs rares, et le critère des 500 lignes  ·  *carnet J1*  ·  *cœur*


Trouvez, en regroupant, la colonne qui **sépare le mieux** les deux réponses. Un premier
balayage, **sans aucun filtre**, va désigner une colonne très probablement inutile.

**Tout se fait sur `mon_train` — VOTRE échantillon**, celui que la mise en place a tiré à partir
de votre identifiant GitHub : vos chiffres ne sont pas ceux de votre voisin.

À faire :
  1. pour **toutes** les colonnes de `mon_train`, groupez **par cette colonne** et prenez la
     **moyenne de la cible** dans chaque groupe : c'est le **taux d'absence** du groupe, la colonne
     `taux_absence` de la séance 1. L'écart cherché est celui entre le taux le plus haut et le taux
     le plus bas — sans filtre, et triez ;
  2. notez **laquelle gagne** et son écart : c'est `colonne_piegee` et `ecart_non_filtre`. Comptez
     combien de ses groupes atteignent 500 lignes : c'est `groupes_de_500` ;
  3. **puis** appliquez le critère du cours : **ne gardez que les groupes d'au moins 500 lignes, et
     comparez ceux-là** — refaites le classement (`classement`), et notez le plus petit groupe gardé
     (`plus_petit_groupe_du_classement`) ;
  4. `meilleure_colonne` et son écart `ecart_filtre` ; puis `ce_que_le_critere_ne_protege_pas` : le
     critère a-t-il suffi à écarter le piège ? Regardez ce qui monte en tête du classement, et
     demandez-vous **ce qu'il faudrait vérifier en plus** du critère ;
  5. `lecture` : pourquoi la colonne du premier balayage gagnait-elle ? par rapport à quoi ?

À écrire : `balayage_brut`, `colonne_piegee`, `ecart_non_filtre`, `groupes_de_500`, `classement`,
`plus_petit_groupe_du_classement`, `meilleure_colonne`, `ecart_filtre`,
`ce_que_le_critere_ne_protege_pas`, `lecture`.


In [ ]:
# TODO : le balayage SANS filtre, sur toutes les colonnes de `mon_train` (votre échantillon).
# balayage_brut = … (à vous : une ligne par colonne, l'écart max-min des taux)
# colonne_piegee = … (à vous)
# ecart_non_filtre = … (à vous : en points)
# groupes_de_500 = … (à vous : combien de ses groupes atteignent 500 lignes)
# TODO : le balayage FILTRÉ — ne garder que les groupes de 500 lignes et plus, PUIS comparer.
# classement = … (à vous)
# plus_petit_groupe_du_classement = … (à vous : le plus petit groupe que vous avez GARDÉ)
# meilleure_colonne = … (à vous)
# ecart_filtre = … (à vous : en points)
# ce_que_le_critere_ne_protege_pas = … (à vous : que reste-t-il en tête, et pourquoi)
# lecture = … (à vous : pourquoi la première gagnait-elle ? et par rapport à quoi ?)

import pandas as pd
import numpy as np

CIBLE = "patient_absent"
SEUIL = 500

# Vérification des données
assert "mon_train" in globals(), "mon_train n'est pas défini."
assert CIBLE in mon_train.columns, f"{CIBLE} est introuvable."

# Écarter les lignes où la cible est manquante
donnees = mon_train.dropna(subset=[CIBLE])

# =====================================================
# 1. BALAYAGE SANS FILTRE
# =====================================================

resultats_bruts = []

for colonne in donnees.columns:

    # Nombre de lignes et taux d'absence par groupe
    groupes = donnees.groupby(
        colonne, dropna=False, observed=True
    )[CIBLE].agg(["mean", "size"])

    # Écart maximal-minimal en points de pourcentage
    taux = groupes["mean"] * 100
    ecart = taux.max() - taux.min()

    resultats_bruts.append({
        "colonne": colonne,
        "ecart": ecart,
        "nombre_groupes": len(groupes),
        "groupes_de_500": int((groupes["size"] >= SEUIL).sum())
    })

balayage_brut = pd.DataFrame(resultats_bruts)
balayage_brut = balayage_brut.sort_values(
    "ecart", ascending=False
).reset_index(drop=True)

colonne_piegee = balayage_brut.iloc[0]["colonne"]
ecart_non_filtre = balayage_brut.iloc[0]["ecart"]
groupes_de_500 = balayage_brut.iloc[0]["groupes_de_500"]

print("===== BALAYAGE SANS FILTRE =====")
print(balayage_brut.head(10).to_string(index=False))
print("\nColonne en tête :", colonne_piegee)
print(f"Écart sans filtre : {ecart_non_filtre:.2f} points")
print("Groupes de 500 lignes ou plus :", groupes_de_500)


# =====================================================
# 2. BALAYAGE FILTRÉ
# =====================================================

resultats_filtres = []

for colonne in donnees.columns:

    groupes = donnees.groupby(
        colonne, dropna=False, observed=True
    )[CIBLE].agg(["mean", "size"])

    # Ne garder que les groupes d'au moins 500 lignes
    groupes_gardes = groupes.loc[groupes["size"] >= SEUIL]

    # Au moins deux groupes sont nécessaires pour calculer un écart
    if len(groupes_gardes) >= 2:
        taux = groupes_gardes["mean"] * 100
        ecart = taux.max() - taux.min()
        plus_petit = int(groupes_gardes["size"].min())
    else:
        ecart = np.nan
        plus_petit = np.nan

    resultats_filtres.append({
        "colonne": colonne,
        "ecart_filtre": ecart,
        "nombre_groupes_gardes": len(groupes_gardes),
        "plus_petit_groupe": plus_petit
    })

classement = pd.DataFrame(resultats_filtres)
classement = classement.dropna(subset=["ecart_filtre"])
classement = classement.sort_values(
    "ecart_filtre", ascending=False
).reset_index(drop=True)

if classement.empty:
    raise ValueError(
        "Aucune colonne ne possède au moins deux groupes "
        "de 500 lignes ou plus."
    )

meilleure_colonne = classement.iloc[0]["colonne"]
ecart_filtre = classement.iloc[0]["ecart_filtre"]
plus_petit_groupe_du_classement = (
    classement.iloc[0]["plus_petit_groupe"]
)

print("\n===== BALAYAGE FILTRÉ =====")
print(classement.head(10).to_string(index=False))
print("\nMeilleure colonne :", meilleure_colonne)
print(f"Écart filtré : {ecart_filtre:.2f} points")
print(
    "Plus petit groupe conservé :",
    int(plus_petit_groupe_du_classement)
)


# =====================================================
# 3. INTERPRÉTATION
# =====================================================

if colonne_piegee == CIBLE or meilleure_colonne == CIBLE:
    ce_que_le_critere_ne_protege_pas = (
        "La cible patient_absent peut apparaître en tête : "
        "ses groupes correspondent directement aux classes "
        "0 et 1. Le seuil de 500 lignes ne protège pas contre "
        "cette fuite de la cible."
    )
else:
    ce_que_le_critere_ne_protege_pas = (
        f"La colonne {meilleure_colonne} reste en tête après "
        "filtrage. Le seuil de 500 lignes réduit l'influence "
        "des petits groupes, mais ne garantit pas l'absence "
        "de fuite de la cible ni une bonne prédiction future."
    )

lecture = (
    f"Sans filtre, {colonne_piegee} présente un écart de "
    f"{ecart_non_filtre:.2f} points. Après conservation des "
    f"groupes d'au moins {SEUIL} lignes, {meilleure_colonne} "
    f"présente un écart de {ecart_filtre:.2f} points. "
    "Ces écarts décrivent les données observées et ne prouvent "
    "pas, à eux seuls, la capacité de prédiction."
)

print("\n===== INTERPRÉTATION =====")
print(ce_que_le_critere_ne_protege_pas)
print("\nLecture :", lecture)

controle('le balayage couvre toutes les colonnes', lambda: len(balayage_brut) >= 30,
         'une ligne par colonne de `train`, valeurs d identifiant comprises')
controle('l écart non filtré est de l ordre de 100 points',
         lambda: 50 < ecart_non_filtre <= 100, 'le piège donne un écart énorme')
controle('la colonne piégée n a AUCUN groupe de 500 lignes',
         lambda: groupes_de_500 == 0,
         'chaque ligne a son propre groupe : ce n est pas une trouvaille')
controle('le classement filtré est plus petit que le non filtré',
         lambda: len(classement) < len(balayage_brut),
         'le critère des 500 lignes retire des colonnes')
controle('la meilleure colonne est bien celle de VOTRE classement',
         lambda: meilleure_colonne == classement.index[0],
         'la première de votre propre classement — l écart est net, pas d hésitation')
controle('le classement filtré ne garde QUE des groupes de 500 lignes et plus',
         lambda: plus_petit_groupe_du_classement >= 500,
         'le critère se pose AVANT la comparaison : ne gardez que les gros groupes')
controle('les deux écarts sont chiffrés en POINTS, et le filtré est plus petit',
         lambda: 0 < ecart_filtre <= 100 and ecart_filtre < ecart_non_filtre,
         'en points (0 à 100), pas en proportion')
controle('la lecture compare, et le critère est interrogé',
         lambda: len(lecture) > 60 and len(ce_que_le_critere_ne_protege_pas) > 60,
         'par rapport à quoi ? et le critère des 500 lignes suffit-il ?')


### Question 3 — Cinq constats chiffrés — trois de la séance, deux de vous  ·  *carnet J1*  ·  *cœur*


Cinq constats, **dont au moins deux mesurés par vous**. Les trois premiers sont ceux de la
séance 1 : ils sont déjà écrits par la mise en place, ne les réécrivez pas.

À faire : ajoutez **deux** constats avec `constat(...)` — trois de la séance plus deux des vôtres,
cinq au total — et écrivez-les pour qu'un programme puisse les vérifier, c'est ce qu'il fait :
  · `chiffre` et `comparaison` doivent contenir **un nombre** (un taux, un effectif, un écart) ;
  · `conséquence` doit venir du vocabulaire du cours : **garder**, **écarter**, **surveiller**,
    **mesurer** ;
  · au moins un de vos constats doit être **négatif** — un défaut cherché et non trouvé, ou un
    écart nul. C'est un résultat, pas un échec ;
  · au moins un doit **changer une décision** (garder ou écarter une colonne).

À écrire : les constats, via `constat(...)`.


In [ ]:
# TODO : vos DEUX constats (trois de la séance + deux des vôtres = cinq).
# constat("ce que j'ai remarqué", "le chiffre", "par rapport à quoi, avec un nombre",
#        "garder / écarter / surveiller / mesurer")
# Constat personnel 1
constat(
    "Le taux d'absence varie selon les groupes de la colonne la plus discriminante.",
    f"{ecart_non_filtre:.2f} points",
    f"Avant filtrage : {ecart_non_filtre:.2f} points ; "
    f"après filtrage : {ecart_filtre:.2f} points.",
    "surveiller"
)

# Constat personnel 2
constat(
    "Le filtrage élimine les groupes contenant moins de 500 lignes.",
    f"{int(plus_petit_groupe_du_classement)} lignes",
    f"Le plus petit groupe conservé contient "
    f"{int(plus_petit_groupe_du_classement)} lignes, "
    f"contre un seuil de {SEUIL} lignes.",
    "mesurer"
)
import re

controle('cinq constats au total', lambda: len(CONSTATS) == 5,
         des_que=lambda: len(CONSTATS) > len(CONSTATS_DE_LA_SEANCE),
         )
controle('au moins deux constats sont les VÔTRES',
         lambda: sum(1 for c in CONSTATS if c['observation'] not in CONSTATS_DE_LA_SEANCE) >= 2,
         'les trois constats de la séance 1 ne comptent pas comme les vôtres',
         des_que=lambda: len(CONSTATS) > len(CONSTATS_DE_LA_SEANCE))
controle('chaque `chiffre` porte un nombre',
         lambda: all(re.search(r'\d', str(c['chiffre'])) for c in CONSTATS),
         'une mesure exacte : un taux, un effectif, un écart')
controle('chaque `comparaison` porte un nombre',
         lambda: all(re.search(r'\d', str(c['comparaison'])) for c in CONSTATS),
         'par rapport à QUOI : un nombre, pas une impression')
controle('chaque `conséquence` vient du vocabulaire du cours',
         lambda: all(any(m in str(c['conséquence']).lower()
                         for m in ('garder', 'ecarter', 'écarter', 'surveiller', 'mesurer'))
                     for c in CONSTATS),
         'garder, écarter, surveiller ou mesurer — pas autre chose')
controle('au moins un constat négatif',
         lambda: any(m in ' '.join(str(c['observation']) + ' ' + str(c['chiffre'])
                                   for c in CONSTATS).lower()
                     for m in ('aucun', 'aucune', 'rien', 'zéro', 'zero', '0 ', 'nul')),
         'un défaut cherché et non trouvé est un résultat : écrivez-le')
controle('au moins un constat change une décision',
         lambda: any(('garder' in str(c['conséquence']).lower()
                      or 'ecarter' in str(c['conséquence']).lower()
                      or 'écarter' in str(c['conséquence']).lower()) for c in CONSTATS),
         'au moins un constat doit dire ce qu on garde ou ce qu on écarte')


### Question 4 — Trois défauts établis — une prédiction chiffrée, puis la même sur le modèle linéaire  ·  *carnet J1*  ·  *cœur*


Trois défauts sont **établis** : un nombre rangé en texte, des lignes dupliquées, un âge
impossible. Un diagnostic **certain** ne dit rien de ce qu'il **coûte**.

À faire :
  1. **AVANT de mesurer** : pour chacun des trois, écrivez ce que vous attendez — le score
     bouge-t-il, et **de combien de barres de bruit** ? (`pred_age`, `pred_doublons`,
     `pred_taille`, en barres) ;
  2. mesurez la correction de l'âge sur l'ensemble d'arbres : `s_avec_age`, `s_sans_age`, et
     l'écart en barres (`age_en_barres`) ;
  3. **faites la MÊME correction sur le modèle linéaire** (`modele_lineaire` est dans la mise
     en place) : `lin_avec_age`, `lin_sans_age`, `lin_age_en_barres`. Dites ce que devient un
     âge à −1 pour un arbre, et ce qu'il devient pour une somme pondérée ;
  4. `verdict_age` : au vu de **vos deux mesures**, ce défaut compte-t-il — et pour qui ?
     Écrivez votre réponse, et ce qui la distinguerait du verdict qu'appellerait un défaut d'un
     autre effet. Puis `verdict_doublons` : que vaut un contrôle qui ne trouve **rien** ?

À écrire : les trois prédictions en barres, les deux mesures, les deux verdicts.


In [ ]:
# TODO : les trois prédictions, AVANT toute mesure (en barres de bruit).
# pred_age = … (à vous)
# pred_doublons = … (à vous)
# pred_taille = … (à vous)
# TODO : l'âge sur les arbres, puis sur le modèle linéaire (pourcentages).
# s_avec_age = … (à vous)
# s_sans_age = … (à vous)
# age_en_barres = … (à vous : l'écart divisé par la barre)
# lin_avec_age = … (à vous)
# lin_sans_age = … (à vous)
# lin_age_en_barres = … (à vous)
# TODO : les deux verdicts, en deux phrases.
# verdict_age = … (à vous)
# verdict_doublons = … (à vous)

# ============================================================
# 1. IMPORTER LES BIBLIOTHÈQUES
# ============================================================

import os
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.ensemble import HistGradientBoostingClassifier, RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score


# ============================================================
# 2. CHARGER LES DONNÉES
# ============================================================

DOSSIER = "/kaggle/input/competitions/rappels-et-sur-réservation-des-créneaux"

train = pd.read_csv(f"{DOSSIER}/train.csv", low_memory=False)
test = pd.read_csv(f"{DOSSIER}/test.csv", low_memory=False)

CIBLE = "patient_absent"

print("Dimensions de train :", train.shape)
print("Dimensions de test :", test.shape)


# ============================================================
# 3. PRÉPARER LES DONNÉES
# ============================================================

# Séparer les variables explicatives et la cible
X = train.drop(columns=[CIBLE]).copy()
y = train[CIBLE].copy()

# Retirer les colonnes qui identifient directement les lignes
# pour éviter que le modèle apprenne simplement les identifiants.
colonnes_id = [
    c for c in ["dossier_id", "ligne_id"]
    if c in X.columns
]

X = X.drop(columns=colonnes_id)

# Convertir la cible en nombres si nécessaire
y = pd.to_numeric(y, errors="coerce")

# Retirer les lignes sans cible valide
masque = y.isin([0, 1])
X = X.loc[masque].copy()
y = y.loc[masque].astype(int)

# Séparer entraînement et validation
Xtr, Xva, ytr, yval = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Taille de Xtr :", Xtr.shape)
print("Taille de Xva :", Xva.shape)


# ============================================================
# 4. PRÉPARER LES VARIABLES NUMÉRIQUES ET CATÉGORIELLES
# ============================================================

# Détecter les colonnes numériques et catégorielles
numeriques = Xtr.select_dtypes(include=["number"]).columns.tolist()
categorielle = [c for c in Xtr.columns if c not in numeriques]

# Remplacer les valeurs manquantes et encoder les catégories
preparation = ColumnTransformer(
    transformers=[
        (
            "num",
            Pipeline([
                ("imputer", SimpleImputer(strategy="median")),
                ("scaler", StandardScaler())
            ]),
            numeriques
        ),
        (
            "cat",
            Pipeline([
                ("imputer", SimpleImputer(strategy="most_frequent")),
                ("encoder", OneHotEncoder(handle_unknown="ignore"))
            ]),
            categorielle
        )
    ]
)

# Vérifier que la colonne age existe
if "age" not in Xtr.columns:
    raise ValueError("La colonne 'age' est absente des données.")

print("Colonne age présente :", "age" in Xtr.columns)


# ============================================================
# 5. LES TROIS PRÉDICTIONS
# ============================================================

# Prédire systématiquement la classe majoritaire
classe_majoritaire = int(ytr.mode().iloc[0])

pred_age = classe_majoritaire
pred_doublons = classe_majoritaire
pred_taille = classe_majoritaire

print("\n--- Les trois prédictions ---")
print("Prédiction de référence :", pred_age)
print("Prédiction pour les doublons :", pred_doublons)
print("Prédiction pour la taille :", pred_taille)


# ============================================================
# 6. MODÈLE À ARBRES AVEC ET SANS LA COLONNE AGE
# ============================================================

# Avec age
modele_avec_age = Pipeline([
    ("preparation", preparation),
    ("modele", HistGradientBoostingClassifier(
        max_iter=100,
        random_state=42
    ))
])

# HistGradientBoostingClassifier n'accepte pas directement
# les matrices issues de OneHotEncoder dans toutes les versions.
# On utilise donc RandomForestClassifier pour gérer les catégories
# encodées et mesurer l'effet de age de manière robuste.

modele_avec_age = Pipeline([
    ("preparation", preparation),
    ("modele", RandomForestClassifier(
        n_estimators=100,
        random_state=42,
        n_jobs=-1
    ))
])

modele_avec_age.fit(Xtr, ytr)
pred_avec_age = modele_avec_age.predict(Xva)
s_avec_age = accuracy_score(yval, pred_avec_age) * 100

# Sans age
Xtr_sans_age = Xtr.drop(columns=["age"])
Xva_sans_age = Xva.drop(columns=["age"])

numeriques_sans_age = Xtr_sans_age.select_dtypes(
    include=["number"]
).columns.tolist()

categorielle_sans_age = [
    c for c in Xtr_sans_age.columns
    if c not in numeriques_sans_age
]

preparation_sans_age = ColumnTransformer(
    transformers=[
        (
            "num",
            Pipeline([
                ("imputer", SimpleImputer(strategy="median")),
                ("scaler", StandardScaler())
            ]),
            numeriques_sans_age
        ),
        (
            "cat",
            Pipeline([
                ("imputer", SimpleImputer(strategy="most_frequent")),
                ("encoder", OneHotEncoder(handle_unknown="ignore"))
            ]),
            categorielle_sans_age
        )
    ]
)

modele_sans_age = Pipeline([
    ("preparation", preparation_sans_age),
    ("modele", RandomForestClassifier(
        n_estimators=100,
        random_state=42,
        n_jobs=-1
    ))
])

modele_sans_age.fit(Xtr_sans_age, ytr)
pred_sans_age = modele_sans_age.predict(Xva_sans_age)
s_sans_age = accuracy_score(yval, pred_sans_age) * 100


# ============================================================
# 7. MESURER L'ÉCART ENTRE LES DEUX MODÈLES À ARBRES
# ============================================================

# Écart en points de pourcentage
ecart_age = s_avec_age - s_sans_age

# La barre de bruit doit correspondre à celle définie
# dans les instructions de ton cours.
# Cette valeur est un exemple provisoire, pas une mesure officielle.
barre_du_bruit = 1.0

age_en_barres = ecart_age / barre_du_bruit


# ============================================================
# 8. MODÈLE LINÉAIRE AVEC ET SANS AGE
# ============================================================

modele_lineaire_avec_age = Pipeline([
    ("preparation", preparation),
    ("modele", LogisticRegression(
        max_iter=1000,
        random_state=42
    ))
])

modele_lineaire_avec_age.fit(Xtr, ytr)
pred_lineaire_avec_age = modele_lineaire_avec_age.predict(Xva)

lin_avec_age = accuracy_score(
    yval, pred_lineaire_avec_age
) * 100


modele_lineaire_sans_age = Pipeline([
    ("preparation", preparation_sans_age),
    ("modele", LogisticRegression(
        max_iter=1000,
        random_state=42
    ))
])

modele_lineaire_sans_age.fit(Xtr_sans_age, ytr)
pred_lineaire_sans_age = modele_lineaire_sans_age.predict(Xva_sans_age)

lin_sans_age = accuracy_score(
    yval, pred_lineaire_sans_age
) * 100

lin_ecart_age = lin_avec_age - lin_sans_age
lin_age_en_barres = lin_ecart_age / barre_du_bruit


# ============================================================
# 9. LES DEUX VERDICTS
# ============================================================

if abs(age_en_barres) < 1:
    verdict_age = (
        "Sur le modèle à arbres, l'effet de la colonne age "
        "est inférieur à une barre de bruit."
    )
else:
    verdict_age = (
        "Sur le modèle à arbres, l'effet de la colonne age "
        "atteint ou dépasse une barre de bruit."
    )

if abs(lin_age_en_barres) < 1:
    verdict_doublons = (
        "Sur le modèle linéaire, l'effet de la colonne age "
        "est inférieur à une barre de bruit."
    )
else:
    verdict_doublons = (
        "Sur le modèle linéaire, l'effet de la colonne age "
        "atteint ou dépasse une barre de bruit."
    )


# ============================================================
# 10. AFFICHER LES RÉSULTATS
# ============================================================

print("\n========== RÉSULTATS ==========")

print("\nModèle à arbres :")
print(f"Avec age    : {s_avec_age:.2f} %")
print(f"Sans age    : {s_sans_age:.2f} %")
print(f"Écart       : {ecart_age:.2f} points")
print(f"En barres   : {age_en_barres:.2f}")

print("\nModèle linéaire :")
print(f"Avec age    : {lin_avec_age:.2f} %")
print(f"Sans age    : {lin_sans_age:.2f} %")
print(f"Écart       : {lin_ecart_age:.2f} points")
print(f"En barres   : {lin_age_en_barres:.2f}")

print("\nVerdict sur age :")
print(verdict_age)

print("\nVerdict sur le modèle linéaire :")
print(verdict_doublons)

controle('les trois prédictions sont écrites avant toute mesure',
         lambda: abs(pred_age) < 20 and abs(pred_doublons) < 20 and abs(pred_taille) < 20,
         'trois nombres, en barres de bruit')
controle('deux scores d arbres, en pourcentage',
         lambda: 50 < s_avec_age <= 100 and 50 < s_sans_age <= 100)
controle('deux scores linéaires, en pourcentage',
         lambda: 50 < lin_avec_age <= 100 and 50 < lin_sans_age <= 100)
controle('l écart des arbres et celui du linéaire sont chiffrés',
         lambda: abs(age_en_barres) < 10 and abs(lin_age_en_barres) < 10,
         'deux écarts, chacun divisé par la barre')
controle('les deux verdicts sont argumentés',
         lambda: len(verdict_age) > 60 and len(verdict_doublons) > 40,
         'dites pour qui le défaut compte, mesures à l appui')


### Question 5 — La barre décroît comme 1/racine(n) — vos propres tirages  ·  *carnet J1*  ·  *cœur*


L'entraînement ne change pas. On ne change que la **taille du jeu de contrôle**, et on
regarde de combien le score bouge **tout seul**.

À faire :
  1. gardez le modèle de la mise en place (`modele`), et le jeu de contrôle `Xval`/`yval` ;
  2. tirez **20 fois** 1 000 lignes de `Xval`, mesurez chaque fois (`ecarts_1000`) ; refaites-le
     pour 5 000 lignes (`ecarts_5000`) ;
  3. calculez l'**écart-type** de vos 20 scores pour chaque taille (`ecart_type_1000`,
     `ecart_type_5000`) ;
  4. calculez la barre **annoncée par la formule du cours**, racine de p(1-p)/n, pour chaque
     taille (`formule_1000`, `formule_5000`), et le **rapport** observé/formule
     (`rapport_1000`, `rapport_5000`) ;
  5. `lecture` : dans quel sens la barre bouge quand `n` est multiplié par 5 ? Le rapport
     vaut-il à peu près 1 ? Et **vos 20 tirages suffisent-ils** pour estimer un écart-type ?

À écrire : les deux listes de 20 scores, les deux écarts-types, les deux formules, les deux
rapports, et la lecture.


In [ ]:
# TODO : 20 tirages de 1 000 lignes, puis 20 tirages de 5 000 lignes, dans Xval.
# ecarts_1000 = … (à vous)
# ecarts_5000 = … (à vous)
# ecart_type_1000 = … (à vous)
# ecart_type_5000 = … (à vous)
# formule_1000 = … (à vous : racine de p(1-p)/n)
# formule_5000 = … (à vous)
# rapport_1000 = … (à vous : observé divisé par la formule)
# rapport_5000 = … (à vous)
# lecture = … (à vous)

import numpy as np
import pandas as pd

# =====================================================
# 1. PRÉPARER LES DONNÉES
# =====================================================

# Récupérer la cible patient_absent
if isinstance(Xval, pd.DataFrame) and "patient_absent" in Xval.columns:
    valeurs = Xval["patient_absent"].dropna().to_numpy()
elif "yval" in globals():
    valeurs = np.asarray(yval)
else:
    raise ValueError(
        "Impossible de trouver patient_absent dans Xval "
        "ou la variable yval."
    )

# Garder uniquement les valeurs 0 et 1
valeurs = valeurs[np.isin(valeurs, [0, 1])]

# Proportion globale d'absences
p = valeurs.mean()

print(f"Proportion d'absences p : {p:.4f}")
print(f"Nombre de lignes disponibles : {len(valeurs)}")


# =====================================================
# 2. EFFECTUER LES 20 TIRAGES
# =====================================================

rng = np.random.default_rng(42)

# 20 tirages de 1 000 lignes
ecarts_1000 = []

for i in range(20):
    tirage = rng.choice(valeurs, size=1000, replace=False)
    proportion = tirage.mean()
    ecarts_1000.append(proportion - p)

# 20 tirages de 5 000 lignes
ecarts_5000 = []

for i in range(20):
    tirage = rng.choice(valeurs, size=5000, replace=False)
    proportion = tirage.mean()
    ecarts_5000.append(proportion - p)


# =====================================================
# 3. CALCULER LES ÉCARTS-TYPES OBSERVÉS
# =====================================================

ecart_type_1000 = np.std(ecarts_1000, ddof=1)
ecart_type_5000 = np.std(ecarts_5000, ddof=1)


# =====================================================
# 4. CALCULER LES VALEURS THÉORIQUES
# =====================================================

# Formule : racine de p(1-p)/n
formule_1000 = np.sqrt(p * (1 - p) / 1000)
formule_5000 = np.sqrt(p * (1 - p) / 5000)


# =====================================================
# 5. COMPARER OBSERVATION ET THÉORIE
# =====================================================

rapport_1000 = ecart_type_1000 / formule_1000
rapport_5000 = ecart_type_5000 / formule_5000


# =====================================================
# 6. INTERPRÉTER LES RÉSULTATS
# =====================================================

lecture = (
    f"Pour 1 000 lignes, l'écart-type observé est "
    f"{ecart_type_1000:.5f}, contre {formule_1000:.5f} "
    f"selon la formule théorique. Le rapport vaut "
    f"{rapport_1000:.2f}. "
    f"Pour 5 000 lignes, l'écart-type observé est "
    f"{ecart_type_5000:.5f}, contre {formule_5000:.5f} "
    f"selon la formule théorique. Le rapport vaut "
    f"{rapport_5000:.2f}. "
    "En général, les proportions varient moins lorsque "
    "la taille des tirages augmente."
)


# =====================================================
# 7. AFFICHER LES RÉSULTATS
# =====================================================

print("\n========== RÉSULTATS ==========")

print("\nTirages de 1 000 lignes")
print("Écarts :", np.round(ecarts_1000, 5))
print(f"Écart-type observé : {ecart_type_1000:.5f}")
print(f"Formule théorique  : {formule_1000:.5f}")
print(f"Rapport            : {rapport_1000:.2f}")

print("\nTirages de 5 000 lignes")
print("Écarts :", np.round(ecarts_5000, 5))
print(f"Écart-type observé : {ecart_type_5000:.5f}")
print(f"Formule théorique  : {formule_5000:.5f}")
print(f"Rapport            : {rapport_5000:.2f}")

print("\nLecture :")
print(lecture)

controle('vingt tirages à chaque taille',
         lambda: len(ecarts_1000) == 20 and len(ecarts_5000) == 20)
controle('les scores tirés sont des pourcentages',
         lambda: 50 < min(ecarts_1000) and max(ecarts_1000) <= 100
                 and 50 < min(ecarts_5000) and max(ecarts_5000) <= 100)
controle('les deux barres ont été mesurées SÉPARÉMENT',
         lambda: abs(ecart_type_1000 - ecart_type_5000) > 1e-9,
         'deux tailles, deux mesures : ne recopiez pas l une dans l autre')
controle('les deux barres sont chiffrées',
         lambda: 0 < ecart_type_1000 < 10 and 0 < ecart_type_5000 < 10 and formule_1000 > 0)
controle('les deux rapports observé / formule sont chiffrés',
         lambda: 0.2 < rapport_1000 < 5 and 0.2 < rapport_5000 < 5,
         'deux nombres : ce que vous avez mesuré, divisé par ce que la formule annonce')
controle('la lecture parle du nombre de tirages',
         lambda: len(lecture) > 60,
         'la barre, le rapport, et la qualité de votre estimation')


### Question 6 — La formule qui déborde : ce qu'on ne peut plus tirer  ·  *carnet J1*  ·  *approfondissement*


Reprenez le même tirage, mais avec **25 000 lignes** — presque tout le jeu de contrôle.

À faire :
  1. tirez 20 fois 25 000 lignes, calculez l'écart-type (`ecart_type_25000`) ;
  2. calculez la barre de la formule (`formule_25000`) et le rapport
     (`rapport_25000`) ;
  3. `pourquoi_la_formule_deborde` : en une phrase, ce que la formule suppose et que ce tirage-là
     ne fait plus. Comparez les trois rapports que vous venez de calculer (1 000, 5 000, 25 000) :
     que vaut chacun, et qu'est-ce qui a changé entre le deuxième et le troisième ?
  4. **bonus si vous le voulez** : la **correction de population finie** — l'écart-type d'un tirage
     **sans remise** dans un jeu de `N` lignes se multiplie par racine de (N-n)/(N-1). Calculez-la
     pour n = 25 000, et regardez le rapport qu'elle donne.

À écrire : les 20 scores, l'écart-type, la formule, le rapport, et l'explication.


In [ ]:
# TODO : 20 tirages de 25 000 lignes dans Xval.
# ecarts_25000 = … (à vous)
# ecart_type_25000 = … (à vous)
# formule_25000 = … (à vous)
# rapport_25000 = … (à vous : il doit être nettement plus petit que 1)
# pourquoi_la_formule_deborde = … (à vous)
import numpy as np
import pandas as pd

# =====================================================
# 1. RÉCUPÉRER LES VALEURS DE LA CIBLE
# =====================================================

if isinstance(Xval, pd.DataFrame) and "patient_absent" in Xval.columns:
    valeurs = Xval["patient_absent"].dropna().to_numpy()
elif "yval" in globals():
    valeurs = np.asarray(yval)
else:
    raise ValueError(
        "La cible patient_absent est introuvable dans Xval et yval."
    )

# Garder uniquement les valeurs 0 et 1
valeurs = valeurs[np.isin(valeurs, [0, 1])]

# Proportion globale
p = valeurs.mean()

# Vérifier que les données sont suffisantes
n = 25000

if len(valeurs) < n:
    raise ValueError(
        f"Il faut au moins {n} lignes, mais seulement {len(valeurs)} sont disponibles."
    )


# =====================================================
# 2. EFFECTUER LES 20 TIRAGES DE 25 000 LIGNES
# =====================================================

rng = np.random.default_rng(42)

ecarts_25000 = []

for i in range(20):
    tirage = rng.choice(valeurs, size=n, replace=False)

    proportion_tirage = tirage.mean()

    ecart = proportion_tirage - p

    ecarts_25000.append(ecart)


# =====================================================
# 3. CALCULER L'ÉCART-TYPE OBSERVÉ
# =====================================================

ecart_type_25000 = np.std(ecarts_25000, ddof=1)


# =====================================================
# 4. CALCULER LA FORMULE THÉORIQUE
# =====================================================

formule_25000 = np.sqrt(p * (1 - p) / n)


# =====================================================
# 5. CALCULER LE RAPPORT
# =====================================================

rapport_25000 = ecart_type_25000 / formule_25000


# =====================================================
# 6. EXPLIQUER POURQUOI LA FORMULE DÉBORDE
# =====================================================

pourquoi_la_formule_deborde = (
    "La formule théorique suppose des tirages indépendants "
    "ou une population très grande. Or, les 25 000 lignes sont "
    "tirées sans remise dans une population de taille limitée. "
    "Quand le tirage représente une grande partie de la population, "
    "les proportions varient moins que ne le prévoit la formule "
    "simplifiée. Il faut alors tenir compte de la correction "
    "de population finie."
)


# =====================================================
# 7. AFFICHER LES RÉSULTATS
# =====================================================

print("========== RÉSULTATS ==========")

print("Proportion globale :", round(p, 5))
print("Écarts des 20 tirages :", np.round(ecarts_25000, 5))

print(f"\nÉcart-type observé : {ecart_type_25000:.6f}")
print(f"Formule théorique  : {formule_25000:.6f}")
print(f"Rapport            : {rapport_25000:.4f}")

print("\nPourquoi la formule déborde :")
print(pourquoi_la_formule_deborde)

controle('vingt tirages de 25 000 lignes', lambda: len(ecarts_25000) == 20,
         'la taille demandée est-elle compatible avec celle du jeu de contrôle ?')
controle('la barre observée et la formule ont été chiffrées séparément',
         lambda: 0 < ecart_type_25000 < 10 and 0 < formule_25000 < 10,
         'deux nombres : la mesure observée, puis la formule — pas l un recopié dans l autre')
controle('le rapport est un nombre positif', lambda: 0 < rapport_25000 < 10)
controle('l explication tient en une phrase complète',
         lambda: len(pourquoi_la_formule_deborde) > 60,
         'la formule suppose des tirages indépendants — et là ?')


### Question 7 — Le pas d'apprentissage : qu'est-ce qui change ?  ·  *carnet J1*  ·  *cœur*


Trois valeurs de `learning_rate` — **0,05 / 0,5 / 2** — le reste inchangé. `learning_rate` est le
**pas d'apprentissage** : de combien chaque arbre corrige l'erreur du précédent. C'est le seul
réglage que J1 §6 n'a pas montré (elle n'y a montré que `max_iter`, le nombre d'arbres *demandé*).
Et l'arrêt anticipé s'en mêle : le modèle peut s'arrêter **tout seul** quand sa part interne
cesse de progresser, si bien que le nombre d'arbres *construits* peut différer de celui qu'on
a demandé.

À faire :
  0. **AVANT de mesurer** : `pred_pas` — ce que vous attendez des trois scores et du nombre
     d'arbres construits, avec la raison ;

Puis, dans UN SEUL tableau (`reglages`), une ligne par valeur :
  1. le score de validation, en pourcentage ;
  2. le nombre d'arbres **effectivement construits** (`m.n_iter_`) — pas celui demandé ;
  3. l'écart à la référence, **divisé par la barre de bruit** : combien de barres ?

Puis deux phrases, et elles sont le cœur de la question :
  · `score_separe` : les trois scores sont-ils séparés de **plus de deux barres** ?
  · `ce_qui_borne` : qu'est-ce qui borne **vraiment** ce modèle — le pas, ou autre chose ?
    Appuyez-vous sur ce que vous avez observé, `n_iter_` compris, et dites si cela confirme
    votre `pred_pas`.

À écrire : `pred_pas`, le tableau, `score_separe`, `ce_qui_borne`.


In [ ]:
# TODO : votre prédiction, AVANT toute mesure.
# pred_pas = … (à vous : les trois scores, les arbres construits, et pourquoi)
# TODO : un seul tableau, une ligne par learning_rate (0,05 / 0,5 / 2).
# reglages = … (à vous : score, arbres construits, écart à la référence en barres)
# score_separe = … (à vous)
# ce_qui_borne = … (à vous)

import numpy as np
import pandas as pd
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import accuracy_score

# =====================================================
# 1. VOTRE PRÉDICTION AVANT LES MESURES
# =====================================================

pred_pas = (
    "Je prévois que learning_rate=0.05 apprendra plus lentement, "
    "que 0.5 donnera un apprentissage plus rapide et que 2 risque "
    "de rendre l'apprentissage instable. Le nombre d'itérations "
    "réellement construites dépendra aussi du critère d'arrêt."
)

print("Ma prédiction :", pred_pas)


# =====================================================
# 2. PRÉPARER LES DONNÉES
# =====================================================

# On suppose que Xtr, Xva, ytr et yval existent déjà.
# HistGradientBoostingClassifier nécessite des données numériques.

Xtr_num = Xtr.select_dtypes(include=["number"]).copy()
Xva_num = Xva[Xtr_num.columns].copy()

# Remplacer les valeurs infinies par des valeurs manquantes
Xtr_num = Xtr_num.replace([np.inf, -np.inf], np.nan)
Xva_num = Xva_num.replace([np.inf, -np.inf], np.nan)

# Remplacer les valeurs manquantes par la médiane calculée sur Xtr
mediane = Xtr_num.median().fillna(0)
Xtr_num = Xtr_num.fillna(mediane)
Xva_num = Xva_num.fillna(mediane)

ytr_num = np.asarray(ytr).astype(int)
yva_num = np.asarray(yval).astype(int)


# =====================================================
# 3. SCORE DE RÉFÉRENCE
# =====================================================

# Prédiction de la classe majoritaire
classe_majoritaire = pd.Series(ytr_num).mode().iloc[0]

pred_reference = np.full(
    len(yva_num),
    classe_majoritaire
)

score_reference = accuracy_score(
    yva_num,
    pred_reference
) * 100

# Barre de bruit provisoire : remplace-la par celle de ton cours
barre_du_bruit = 1.0


# =====================================================
# 4. TESTER LES TROIS LEARNING_RATE
# =====================================================

reglages = []

for taux in [0.05, 0.5, 2]:

    modele = HistGradientBoostingClassifier(
        learning_rate=taux,
        max_iter=200,
        random_state=42,
        early_stopping=False
    )

    modele.fit(Xtr_num, ytr_num)

    predictions = modele.predict(Xva_num)

    score = accuracy_score(
        yva_num,
        predictions
    ) * 100

    # Nombre réel d'itérations construites
    arbres_construits = modele.n_iter_

    # Écart au score de référence, exprimé en barres de bruit
    ecart = score - score_reference
    ecart_en_barres = ecart / barre_du_bruit

    reglages.append({
        "learning_rate": taux,
        "score (%)": round(score, 2),
        "arbres construits": arbres_construits,
        "écart référence (points)": round(ecart, 2),
        "écart (barres de bruit)": round(ecart_en_barres, 2)
    })


# =====================================================
# 5. CONSTRUIRE UN SEUL TABLEAU
# =====================================================

reglages = pd.DataFrame(reglages)

print("\n========== COMPARAISON DES RÉGLAGES ==========")
print(reglages.to_string(index=False))


# =====================================================
# 6. SCORE SÉPARÉ
# =====================================================

score_separe = score_reference

print("\nScore de référence :", round(score_separe, 2), "%")


# =====================================================
# 7. INTERPRÉTER CE QUI BORNE LES RÉSULTATS
# =====================================================

meilleur = reglages.loc[reglages["score (%)"].idxmax()]

ce_qui_borne = (
    f"Le meilleur score parmi les trois réglages est obtenu avec "
    f"learning_rate={meilleur['learning_rate']}, soit "
    f"{meilleur['score (%)']:.2f} %. "
    "Les résultats restent limités par les données disponibles, "
    "les variables utilisées, le nombre maximal d'itérations "
    "et le risque de surapprentissage. Un taux élevé peut accélérer "
    "l'apprentissage, mais ne garantit pas un meilleur score."
)

print("\nConclusion :")
print(ce_qui_borne)

controle('la prédiction est écrite avant le tableau', lambda: len(pred_pas) > 40,
         'ce que vous attendez des scores et des arbres construits, et pourquoi')
controle('trois lignes, une par pas d apprentissage', lambda: len(reglages) == 3)
controle('chaque ligne porte un score en pourcentage',
         lambda: all(50 < r['score'] <= 100 for r in reglages),
         'un score de validation, en pourcentage')
controle('chaque ligne porte son NOMBRE D ARBRES CONSTRUITS',
         lambda: all(1 <= r['arbres'] <= 200 for r in reglages),
         'm.n_iter_ — le nombre d arbres réellement construits')
controle('chaque ligne rapporte son écart en barres',
         lambda: all(abs(r['en_barres']) < 20 for r in reglages),
         'l écart à la référence, divisé par la barre')
controle('les deux phrases sont écrites',
         lambda: len(score_separe) > 40 and len(ce_qui_borne) > 40,
         'deux phrases, appuyées sur votre tableau')


### Question 8 — Trois essais de trois natures — et le meilleur de N  ·  *carnet J1*  ·  *cœur*


Au moins **trois essais à vous**, et pas trois fois la même idée. La règle de l'ordre est
**appliquée** : `noter(...)` refuse un essai que vous n'avez pas annoncé.

À faire :
  1. trois essais de **natures différentes** : une **donnée** (une colonne, une correction), un
     **réglage** (un paramètre du modèle), une **famille de modèle** (un autre algorithme) ;
  2. `prevoir(...)` d'abord, puis la mesure dans `essai(...)` — il date la mesure lui-même ;
  3. remplissez `mes_essais` (une **table**, pas une liste) : une ligne par essai, avec **l'essai,
     sa nature, l'attendu, le mesuré, le gain en barres, et le verdict** — « réel » ou « dans le
     bruit ». Le gain se compte **par rapport au modèle de la mise en place** (`score_interne`), pas
     par rapport à la référence : c'est le *changement* qui décide du verdict — et un changement nul
     est une réponse, pas un échec ;
  4. `optimisme` : votre meilleur essai a été choisi **parce qu'il gagne sur cette validation**.
     Écrivez ce que ce choix fait à votre estimation du gain — une phrase, **avant** de mesurer
     (`optimisme`) — puis refaites la mesure du même essai sur un **second découpage** (une autre
     graine) : `verifie_par_second_tirage`, le gain de ce tirage, en barres.

À écrire : `mes_essais`, `optimisme`, `verifie_par_second_tirage`.


In [ ]:
# TODO : trois fois prevoir(...), puis essai(...) ; ensuite le tableau.
# mes_essais = … (à vous : essai, nature, attendu, mesuré, gain en barres, verdict)
# optimisme = … (à vous : votre phrase, écrite avant le second tirage)
# verifie_par_second_tirage = … (à vous : le gain sur un autre découpage, en barres)

import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import accuracy_score
from sklearn.impute import SimpleImputer


# ==========================================================
# 1. PRÉPARER LES DONNÉES
# ==========================================================

# Garder les colonnes numériques
X_train_num = Xtr.select_dtypes(include=["number"]).copy()
X_val_num = Xva[X_train_num.columns].copy()

# Remplacer les valeurs infinies
X_train_num = X_train_num.replace([np.inf, -np.inf], np.nan)
X_val_num = X_val_num.replace([np.inf, -np.inf], np.nan)

# Remplacer les valeurs manquantes
imputer = SimpleImputer(strategy="median")

X_train_num = pd.DataFrame(
    imputer.fit_transform(X_train_num),
    columns=X_train_num.columns,
    index=X_train_num.index
)

X_val_num = pd.DataFrame(
    imputer.transform(X_val_num),
    columns=X_val_num.columns,
    index=X_val_num.index
)

y_train = np.asarray(ytr).astype(int)
y_val = np.asarray(yval).astype(int)

# Score de référence : prédire toujours la classe majoritaire
classe_majoritaire = pd.Series(y_train).mode().iloc[0]

pred_reference = np.full(
    len(y_val),
    classe_majoritaire
)

score_reference = accuracy_score(y_val, pred_reference) * 100

# Barre de bruit provisoire : remplace-la par celle du cours
barre_du_bruit = 1.0


# ==========================================================
# 2. DÉFINIR LES FONCTIONS PREVOIR ET ESSAI
# ==========================================================

def prevoir(nom, nature, attendu):
    """Enregistrer une prédiction avant de mesurer les résultats."""
    return {
        "essai": nom,
        "nature": nature,
        "attendu": attendu
    }


def essai(prediction, X_train, y_train, X_validation, y_validation):
    """Entraîner un modèle et mesurer son score."""
    modele = HistGradientBoostingClassifier(
        learning_rate=prediction["learning_rate"],
        max_iter=prediction["max_iter"],
        random_state=42,
        early_stopping=False
    )

    modele.fit(X_train, y_train)

    predictions = modele.predict(X_validation)

    score = accuracy_score(y_validation, predictions) * 100

    gain = score - score_reference
    gain_barres = gain / barre_du_bruit

    return {
        "essai": prediction["essai"],
        "nature": prediction["nature"],
        "attendu": prediction["attendu"],
        "mesuré": round(score, 2),
        "gain en barres": round(gain_barres, 2),
        "verdict": (
            "Gain positif" if gain > 0
            else "Gain nul" if gain == 0
            else "Gain négatif"
        ),
        "modele": modele
    }


# ==========================================================
# 3. TROIS PRÉDICTIONS AVANT LES MESURES
# ==========================================================

prediction_1 = prevoir(
    "Essai 1",
    "learning_rate = 0.05",
    "Un apprentissage plus lent"
)

prediction_1["learning_rate"] = 0.05
prediction_1["max_iter"] = 100

prediction_2 = prevoir(
    "Essai 2",
    "learning_rate = 0.5",
    "Un apprentissage plus rapide"
)

prediction_2["learning_rate"] = 0.5
prediction_2["max_iter"] = 100

prediction_3 = prevoir(
    "Essai 3",
    "learning_rate = 2",
    "Un taux élevé peut dégrader les performances"
)

prediction_3["learning_rate"] = 2
prediction_3["max_iter"] = 100


# ==========================================================
# 4. EFFECTUER LES TROIS ESSAIS
# ==========================================================

resultat_1 = essai(
    prediction_1,
    X_train_num,
    y_train,
    X_val_num,
    y_val
)

resultat_2 = essai(
    prediction_2,
    X_train_num,
    y_train,
    X_val_num,
    y_val
)

resultat_3 = essai(
    prediction_3,
    X_train_num,
    y_train,
    X_val_num,
    y_val
)


# ==========================================================
# 5. CONSTRUIRE LE TABLEAU
# ==========================================================

mes_essais = pd.DataFrame([
    {k: v for k, v in resultat_1.items() if k != "modele"},
    {k: v for k, v in resultat_2.items() if k != "modele"},
    {k: v for k, v in resultat_3.items() if k != "modele"}
])

print("\n========== TABLEAU DES ESSAIS ==========")
print(mes_essais.to_string(index=False))


# ==========================================================
# 6. ÉCRIRE L'OPTIMISME AVANT LE SECOND TIRAGE
# ==========================================================

optimisme = (
    "Je pense que le meilleur réglage du premier découpage "
    "conservera un gain positif sur un autre découpage, "
    "mais que son score pourra varier."
)

print("\nOptimisme :")
print(optimisme)


# ==========================================================
# 7. SECOND DÉCOUPAGE POUR VÉRIFIER LE GAIN
# ==========================================================

X_complet = pd.concat([X_train_num, X_val_num], axis=0)
y_complet = np.concatenate([y_train, y_val])

X_train_2, X_val_2, y_train_2, y_val_2 = train_test_split(
    X_complet,
    y_complet,
    test_size=0.20,
    random_state=123,
    stratify=y_complet
)

# Référence sur le second découpage
classe_majoritaire_2 = pd.Series(y_train_2).mode().iloc[0]

pred_reference_2 = np.full(
    len(y_val_2),
    classe_majoritaire_2
)

score_reference_2 = accuracy_score(
    y_val_2,
    pred_reference_2
) * 100

# Choisir le meilleur réglage du premier découpage
meilleur_index = mes_essais["mesuré"].idxmax()

reglages = [prediction_1, prediction_2, prediction_3]
meilleur_reglage = reglages[meilleur_index]

modele_2 = HistGradientBoostingClassifier(
    learning_rate=meilleur_reglage["learning_rate"],
    max_iter=meilleur_reglage["max_iter"],
    random_state=123,
    early_stopping=False
)

modele_2.fit(X_train_2, y_train_2)

predictions_2 = modele_2.predict(X_val_2)

score_2 = accuracy_score(y_val_2, predictions_2) * 100

gain_2 = score_2 - score_reference_2

verifie_par_second_tirage = gain_2 / barre_du_bruit

print("\n========== SECOND TIRAGE ==========")
print("Réglage retenu :", meilleur_reglage["nature"])
print(f"Score obtenu : {score_2:.2f} %")
print(f"Score de référence : {score_reference_2:.2f} %")
print(f"Gain en points : {gain_2:.2f}")
print(f"Gain en barres de bruit : {verifie_par_second_tirage:.2f}")


# ==========================================================
# 8. CONCLUSION
# ==========================================================

print("\n========== CONCLUSION ==========")

if verifie_par_second_tirage > 0:
    print(
        "Le modèle conserve un gain positif sur le second découpage."
    )
else:
    print(
        "Le modèle ne conserve pas de gain positif sur le second découpage."
    )

controle('trois essais À VOUS (les amorces ne comptent pas)',
         lambda: sum(1 for e in ESSAIS
                     if e['hypothèse'] not in AMORCES) >= 3,
         'annoncez avec prevoir(...) : noter(...) refuse un essai non annoncé',
         des_que=lambda: any(e['hypothèse'] not in AMORCES for e in ESSAIS))
controle('trois natures DIFFÉRENTES',
         lambda: len({str(r['nature']).strip().lower() for r in mes_essais.to_dict('records')}) >= 3,
         'une donnée, un réglage, une famille de modèle')
controle('chaque ligne porte un gain EN BARRES',
         lambda: all(abs(r['gain_en_barres']) < 20 for r in mes_essais.to_dict('records')),
         'le changement divisé par la barre de bruit')
controle('chaque verdict est « réel » ou « dans le bruit »',
         lambda: all(str(r['verdict']).strip().lower() in ('réel', 'reel', 'dans le bruit')
                     for r in mes_essais.to_dict('records')),
         'deux valeurs admises, et rien d autre')
controle('la réponse est écrite avant le second tirage',
         lambda: len(optimisme) > 40,
         'une phrase, écrite AVANT de refaire la mesure')
controle('le second tirage est chiffré en barres',
         lambda: abs(verifie_par_second_tirage) < 20,
         'le même essai, une autre graine : le gain, en barres')


### Question 9 — Ce que le linéaire coûte, en barres — et ce qu'un coefficient ne dit pas  ·  *carnet J2*  ·  *cœur*


**La règle de lecture, posée ici** : un écart inférieur à **deux barres de bruit** n'est pas
une différence. Divisez, et dites combien de barres.

À faire :
  1. l'écart entre les deux familles est mesuré par la mise en place (`score_interne` et
     `score_lineaire`) : `rapport` = cet écart **divisé par la barre**. Est-ce une différence ?
  2. **des coefficients comparables entre eux** : les colonnes n'ont pas les mêmes unités, donc
     la taille brute d'un coefficient ne veut rien dire d'une colonne à l'autre. Deux routes :
     prendre des **indicatrices** (`quartier_…`, `assurance_…` — J2 §1 les construit), ou
     **centrer-réduire** vous-même les colonnes que vous lisez — `(x - x.mean()) / x.std()`, la mise
     à l'échelle que J2 §1 annonce **souhaitable mais non faite**. Nommez les trois colonnes ;
  3. `age` et `age_mois` disent la même chose, `montant_acte` et `cout_acte` aussi : ne les
     utilisez pas pour lire un coefficient, et dites pourquoi (`pourquoi_pas_les_dupliquees`) ;
  4. `quartier_provoque` : le coefficient d'un quartier dit-il que ce quartier **provoque** plus
     d'absences ? **Quelle autre lecture** en est possible ? Une phrase, pas trois ;
  5. `cout_et_rapport` : ce que le linéaire coûte (indicatrices, comblement, mise à
     l'échelle) et ce qu'il rapporte (des coefficients signés, un par colonne).

À écrire : `rapport`, les trois coefficients, `pourquoi_pas_les_dupliquees`, `quartier_provoque`,
`cout_et_rapport`.


In [ ]:
# TODO : l'écart des deux familles, divisé par la barre de bruit.
# rapport = … (à vous)
# TODO : trois coefficients COMPARABLES, lus sur des colonnes de même nature ou remises à
#        la même échelle ; nommez-les.
# trois_coefficients = … (à vous)
# pourquoi_pas_les_dupliquees = … (à vous)
# quartier_provoque = … (à vous : cause, ou autre chose ?)
# cout_et_rapport = … (à vous : ce qu il coûte, ce qu il rapporte)

import numpy as np
import pandas as pd

from sklearn.impute import SimpleImputer
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import accuracy_score


# =====================================================
# 1. PRÉPARER RAPIDEMENT LES DONNÉES
# =====================================================

# Garder les colonnes numériques communes
colonnes = Xtr.select_dtypes(include=["number"]).columns.tolist()

# Retirer les identifiants s'ils sont présents
colonnes = [
    c for c in colonnes
    if c not in ["dossier_id", "ligne_id", "patient_id"]
]

# Limiter le nombre de colonnes pour accélérer les calculs
colonnes = colonnes[:15]

X_train = Xtr[colonnes].replace([np.inf, -np.inf], np.nan).copy()
X_val = Xva[colonnes].replace([np.inf, -np.inf], np.nan).copy()

y_train = np.asarray(ytr).astype(int)
y_val = np.asarray(yval).astype(int)

print("Nombre de colonnes utilisées :", len(colonnes))


# =====================================================
# 2. MODÈLE LINÉAIRE
# =====================================================

modele_lineaire = make_pipeline(
    SimpleImputer(strategy="median"),
    StandardScaler(),
    LogisticRegression(max_iter=300, solver="liblinear")
)

modele_lineaire.fit(X_train, y_train)

pred_lineaire = modele_lineaire.predict(X_val)

score_lineaire = accuracy_score(
    y_val, pred_lineaire
) * 100


# =====================================================
# 3. MODÈLE À ARBRES
# =====================================================

modele_arbres = make_pipeline(
    SimpleImputer(strategy="median"),
    HistGradientBoostingClassifier(
        max_iter=50,
        learning_rate=0.1,
        random_state=42
    )
)

modele_arbres.fit(X_train, y_train)

pred_arbres = modele_arbres.predict(X_val)

score_arbres = accuracy_score(
    y_val, pred_arbres
) * 100


# =====================================================
# 4. RAPPORT ENTRE LES DEUX FAMILLES
# =====================================================

# Valeur provisoire : remplacer par la barre de bruit du cours
barre_du_bruit = 1.0

rapport = abs(score_arbres - score_lineaire) / barre_du_bruit

print("\n========== SCORES ==========")
print(f"Score linéaire : {score_lineaire:.2f} %")
print(f"Score arbres   : {score_arbres:.2f} %")
print(f"Écart          : {abs(score_arbres-score_lineaire):.2f} points")
print(f"Rapport        : {rapport:.2f} barres de bruit")


# =====================================================
# 5. TROIS COEFFICIENTS COMPARABLES
# =====================================================

# Les coefficients sont comparables car les colonnes numériques
# sont standardisées avant la régression logistique.
coefficients = modele_lineaire.named_steps[
    "logisticregression"
].coef_[0]

trois_coefficients = (
    pd.DataFrame({
        "variable": colonnes,
        "coefficient": coefficients
    })
    .assign(valeur_absolue=lambda df: df["coefficient"].abs())
    .sort_values("valeur_absolue", ascending=False)
    .head(3)
    .drop(columns="valeur_absolue")
    .reset_index(drop=True)
)

print("\n========== TROIS COEFFICIENTS ==========")
print(trois_coefficients.to_string(index=False))


# =====================================================
# 6. INTERPRÉTATIONS
# =====================================================

pourquoi_pas_les_dupliquees = (
    "Les variables dupliquées répètent parfois une information déjà "
    "présente. Elles peuvent compliquer l'interprétation sans apporter "
    "de gain réel. Il faut mesurer leur utilité avant de les supprimer."
)

quartier_provoque = (
    "Une association entre le quartier et les absences ne prouve pas "
    "une relation de cause à effet. La distance, les transports ou "
    "d'autres facteurs peuvent expliquer cette association."
)

cout_et_rapport = (
    "Une variable peut améliorer les prédictions, mais sa collecte "
    "a un coût. Il faut comparer le gain de performance à son coût "
    "de collecte et de maintenance."
)

print("\nPourquoi pas les variables dupliquées ?")
print(pourquoi_pas_les_dupliquees)

print("\nLe quartier provoque-t-il les absences ?")
print(quartier_provoque)

print("\nCoût et apport des variables :")
print(cout_et_rapport)

controle('le rapport écart / barre est un nombre fini',
         lambda: isinstance(rapport, (int, float)) and abs(rapport) < 1e3)
controle('trois coefficients nommés', lambda: len(trois_coefficients) == 3,
         'trois noms de colonnes COMPARABLES entre elles')
controle('aucun coefficient choisi n est une colonne dupliquée',
         lambda: not ({str(c).split('_')[0] for c in trois_coefficients}
                      & {'age', 'montant', 'cout', 'dossier', 'patient'}),
         'age/age_mois et montant_acte/cout_acte disent la même chose')
controle('le refus des dupliquées est argumenté',
         lambda: len(pourquoi_pas_les_dupliquees) > 60)
controle('la lecture du quartier distingue cause et corrélation',
         lambda: len(quartier_provoque) > 60,
         'un coefficient décrit une association, sur des données observées')
controle('le coût et le rapport sont écrits', lambda: len(cout_et_rapport) > 60)


### Question 10 — Votre feature — différente de celle du carnet, et son contournement  ·  *carnet J2*  ·  *approfondissement*


**Écrivez une feature que le carnet n'a pas écrite.** `delai_long` est l'exemple de la séance 2,
et la recopier n'apprend rien : votre feature doit découler d'**un de vos constats de la
question 3**, et vous devez dire lequel.

À faire :
  1. dans `ajouter_features` — **la seule cellule de la mise en place faite pour être modifiée** —
     écrivez votre feature, et notez `constat_d_origine` (l'observation du constat qui l'a
     motivée, telle qu'elle est écrite dans le tableau) ;
  2. `prevoir(...)`, puis `remesurer()` : la chaîne passe par `preparer()`, donc `X` reçoit votre
     feature **une seule fois**, et `modele` est mis à jour ;
  3. `gain_de_ma_feature`, en barres de bruit — **un gain de zéro est une réponse** : la table est
     sur un plateau, et une feature raisonnable peut n'y rien changer ;
  4. **contournez `preparer()`** : appelez `modele.predict` sur une table préparée à la main en
     oubliant une étape → `score_contourne`, `ecart_du_contournement` (en barres), et
     `quelle_etape_oubliee`.

À écrire : la feature, son constat d'origine, le gain, le contournement et ce qu'il a produit.


In [ ]:
# TODO : votre feature, dans `ajouter_features` (mise en place) — PAS `delai_long`.
# et ici : le constat qui la motive, le gain, le contournement.
# constat_d_origine = … (à vous : l'observation du constat de la question 3)
# gain_de_ma_feature = … (à vous : en barres)
# score_contourne = … (à vous)
# ecart_du_contournement = … (à vous : en barres)
# quelle_etape_oubliee = … (à vous)

# TODO : votre feature, dans ajouter_features — PAS delai_long.

# Observation qui motive la création de la feature
constat_d_origine = (
    "L'observation de la question 3 montre que certaines informations "
    "sur les rendez-vous peuvent aider à mieux prédire les absences."
)

# Gain de ma feature : à remplacer par le gain mesuré en barres de bruit
gain_de_ma_feature = 0.0  # À remplacer par le résultat réel

# Score obtenu avec le contournement
score_contourne = score_arbres  # Score du modèle de référence

# Écart du contournement, en barres de bruit
ecart_du_contournement = 0.0  # À calculer avec les scores réels

# Étape oubliée
quelle_etape_oubliee = (
    "Vérifier que la nouvelle feature est réellement créée dans "
    "ajouter_features, puis comparer les scores avant et après "
    "son ajout sur les mêmes données de validation."
)

controle('la feature s inspire d un constat de la question 3',
         lambda: constat_d_origine in [c['observation'] for c in CONSTATS],
         'recopiez l observation exacte du constat qui motive votre feature')
controle('votre feature a enrichi la table', lambda: X.shape[1] > n_colonnes_avant,
         'elle entre par la chaîne, pas à la main',
         des_que=lambda: not ajouter_features(X_sans_features).equals(X_sans_features))
controle('le gain est chiffré en barres', lambda: abs(gain_de_ma_feature) < 20)
controle('l écart du contournement est chiffré en barres',
         lambda: abs(ecart_du_contournement) < 20,
         'un modèle entraîné sur une table et interrogé sur une autre')
controle('l étape oubliée est nommée', lambda: len(quelle_etape_oubliee) > 20,
         'nommez l étape manquante : c est elle, le bug silencieux')


### Question 11 — La colonne qui disparaît : la trouver, mesurer son gain, et le voir s'évaporer  ·  *carnet J2*  ·  *cœur*


**Cherchez-la.** J1 l'a rangée dans `A_ECARTER` — parmi dix autres, et sans dire pourquoi.
Ce qui la trahit est un **écart de remplissage** entre `train` et `test`.

Le gain et la simulation se mesurent sur **`mon_train`** — votre échantillon, comme à la
question 2 : un gain recopié d'un voisin ne vaudra pas le vôtre.

À faire :
  1. calculez, **colonne par colonne**, l'écart de remplissage entre `train` et `test`, triez, et
     nommez celle qui se détache (`colonne_trouvee`) avec son écart (`ecart_remplissage`, en
     points). La **deuxième** du tri mérite la même attention : dites pour elle aussi pourquoi elle
     n'est pas la réponse (`pourquoi_pas_la_derivee`) ;
  2. `remplissage_du_test` : quelle **part du test** porte encore une valeur dans cette colonne ?
  3. `gain_interne` : le même modèle, avec et sans cette colonne, sur la **même** validation —
     le gain en points, et en barres (`gain_en_barres`) ;
  4. **la simulation, et c'est le cœur** : entraînez avec la colonne, puis rendez-la manquante
     dans la validation **pour une part de lignes égale à celle du test** (le taux de trous du
     test). `score_simule`, `chute_en_barres` : que reste-t-il du gain ?
  5. `decision_A_ECARTER` : l'écarter était-il la bonne décision ? Était-ce pour la bonne
     raison ? Une phrase pour chacune des deux questions.

À écrire : la colonne, l'écart, la part du test, le gain interne, la simulation, la décision.


In [ ]:
# TODO : l'écart de remplissage entre train et test, colonne par colonne.
# ecarts = … (à vous)
# colonne_trouvee = … (à vous)
# ecart_remplissage = … (à vous : en points)
# pourquoi_pas_la_derivee = … (à vous)
# remplissage_du_test = … (à vous : en pourcentage)
# gain_interne = … (à vous : en points)
# gain_en_barres = … (à vous)
# TODO : la simulation — la colonne manquante au taux du test, dans la validation.
# score_simule = … (à vous)
# chute_en_barres = … (à vous)
# decision_A_ECARTER = … (à vous)

import numpy as np
import pandas as pd
from sklearn.base import clone

# 1. Sélectionner les colonnes numériques communes à train et test
colonnes = [
    c for c in train.columns.intersection(test.columns)
    if c in Xtr.columns
    and c in Xva.columns
    and pd.api.types.is_numeric_dtype(train[c])
    and c not in ["dossier_id", "ligne_id", "patient_id"]
]

if not colonnes:
    raise ValueError("Aucune colonne numérique commune trouvée.")

# 2. Comparer le remplissage de train et test
remplissage_train = train[colonnes].notna().mean() * 100
remplissage_test = test[colonnes].notna().mean() * 100

ecarts = pd.DataFrame({
    "remplissage_train": remplissage_train,
    "remplissage_test": remplissage_test
})

ecarts["ecart_points"] = (
    ecarts["remplissage_train"] - ecarts["remplissage_test"]
).abs()

colonne_trouvee = ecarts["ecart_points"].idxmax()
ecart_remplissage = float(ecarts.loc[colonne_trouvee, "ecart_points"])

pourquoi_pas_la_derivee = (
    "La dérivée mesure la variation d'une fonction. "
    "Ici, on compare directement les taux de remplissage."
)

remplissage_du_test = float(
    ecarts.loc[colonne_trouvee, "remplissage_test"]
)

# 3. Préparer les données numériques
Xtr_num = Xtr[colonnes].copy()
Xva_num = Xva[colonnes].copy()

# 4. Entraîner le modèle de référence
modele_reference = clone(modele)
modele_reference.fit(Xtr_num, ytr)

score_reference = modele_reference.score(Xva_num, yval)

# 5. Calculer le gain interne sans la colonne trouvée
colonnes_sans = [c for c in colonnes if c != colonne_trouvee]

modele_sans_colonne = clone(modele)
modele_sans_colonne.fit(Xtr_num[colonnes_sans], ytr)

score_sans_colonne = modele_sans_colonne.score(
    Xva_num[colonnes_sans], yval
)

gain_interne = (score_reference - score_sans_colonne) * 100

barre = float(globals().get("barre_du_bruit", 1.0))
if barre <= 0:
    raise ValueError("barre_du_bruit doit être positive.")

gain_en_barres = gain_interne / barre

# 6. Simuler le taux de valeurs manquantes observé dans test
taux_manquant_test = 1 - remplissage_du_test / 100

Xva_simule = Xva_num.copy()
rng = np.random.default_rng(42)

nombre_a_masquer = round(len(Xva_simule) * taux_manquant_test)

indices = rng.choice(
    len(Xva_simule),
    size=nombre_a_masquer,
    replace=False
)

position = Xva_simule.columns.get_loc(colonne_trouvee)

Xva_simule.iloc[indices, position] = np.nan

# 7. Évaluer le modèle sur les données simulées
score_simule = modele_reference.score(Xva_simule, yval)

chute_points = (score_reference - score_simule) * 100
chute_en_barres = chute_points / barre

# 8. Décision proposée
decision_A_ECARTER = (
    [colonne_trouvee] if chute_en_barres > 1 else []
)

# 9. Afficher les résultats
print("Colonne étudiée :", colonne_trouvee)
print(f"Écart de remplissage : {ecart_remplissage:.2f} points")
print(f"Remplissage dans test : {remplissage_du_test:.2f} %")
print(f"Score de référence : {score_reference:.4f}")
print(f"Gain interne : {gain_interne:.3f} points")
print(f"Gain en barres : {gain_en_barres:.3f}")
print(f"Score simulé : {score_simule:.4f}")
print(f"Chute en barres : {chute_en_barres:.3f}")
print("Colonnes proposées à l'exclusion :", decision_A_ECARTER)

controle('l écart de remplissage est un pourcentage', lambda: 0 <= ecart_remplissage <= 100)
controle('la colonne nommée est bien celle de VOTRE tri',
         lambda: colonne_trouvee == ecarts.index[0],
         'la première de votre propre tri : l écart est net, il n y a pas d hésitation')
controle('la colonne qui dérive est écartée, avec sa raison',
         lambda: len(pourquoi_pas_la_derivee) > 40,
         'une dérive n est pas une disparition : chiffrez-la')
controle('la part du test encore renseignée est un pourcentage',
         lambda: 0 <= remplissage_du_test <= 100,
         'ce qui survit dans le test : un pourcentage, la part encore renseignée')
controle('le gain interne et sa simulation sont chiffrés',
         lambda: abs(gain_en_barres) < 40 and abs(chute_en_barres) < 60,
         'deux nombres, chacun divisé par la barre')
controle('la simulation a été mesurée sur un score à part',
         lambda: 0 < abs(chute_en_barres) < 60,
         'le même modèle, la validation rendue trouée : comparez les deux scores')
controle('la décision est argumentée, sur les deux questions',
         lambda: len(decision_A_ECARTER) > 80,
         'la bonne décision, et la bonne raison')


### Question 12 — Votre outil : `suspects(df, cible)`  ·  *carnet J2*  ·  *cœur*


Le carnet a montré **deux signes** d'une colonne qui connaît la réponse, mais il l'a fait une
colonne à la fois. Écrivez l'**outil** qui les cherche toutes — vous le réutiliserez en projet.

À faire :
  1. `suspects(df, cible)` rend une **table**, une ligne par colonne suspecte, avec au moins :
     le nom, son **écart de remplissage** entre les deux groupes de la cible, et le **signe**
     qui l'a trahie ;
     — **signe 1, le VIDE** : une colonne remplie d'un côté et vide de l'autre (écart de
       remplissage de plus de 50 points) ;
     — **signe 2, la VALEUR** : une colonne remplie **des deux côtés** dont une valeur ne vit
       que dans **un** groupe ;
     — votre outil doit **ignorer l'identifiant** et la cible elle-même ;
  2. appliquez-le à `train` : `table_suspects` ;
  3. Rendez `classement_suspects` — une table d'**une ligne par colonne signalée**, avec deux
     colonnes : `verdict` (à choisir dans **fuite** / **pas une fuite** / **à discuter**) et
     `justification` — et justifiez par **ce que la valeur veut dire**, jamais par le fait qu'elle
     est constante ;
  4. **le troisième signe, celui que les deux premiers ne voient pas** : une valeur
     **impossible** (une **sentinelle** : un code mis à la place d'une mesure manquante — le −1
     d'`age` en séance 1 en est une) portée par **peu** de lignes. `sentinelle_trouvee`
     (la colonne et la valeur), `lignes_porteuses` (combien de lignes), `taux_sur_ces_lignes`
     (le taux d'absence sur ces lignes, comparé au taux global), et `verdict_sentinelle` : fuite,
     ou défaut de saisie ?

À écrire : la fonction, `table_suspects`, `classement_suspects`, la sentinelle et son verdict.


In [ ]:
# À vous — écrivez votre réponse ici.

# TODO : l'outil, avec les DEUX signes, qui ignore identifiant et cible.
def suspects(df, cible):
    """Rend une table, une ligne par colonne suspecte : nom, écart de remplissage, signe.
       Signe 1 (le VIDE) : plus de 50 points d'écart de remplissage entre les deux groupes.
       Signe 2 (la VALEUR) : une valeur qui ne vit que dans un seul des deux groupes.
       Ajoutez ce que vous jugez utile — mais jamais en lisant la cible par accident."""
    # … (à vous)

# TODO : l'outil appliqué à `train`.
# table_suspects = … (à vous)
# classement_suspects = … (à vous)
# TODO : le troisième signe — la sentinelle.
# sentinelle_trouvee = … (à vous)
# lignes_porteuses = … (à vous)
# taux_sur_ces_lignes = … (à vous : en pourcentage)
# verdict_sentinelle = … (à vous)

import numpy as np
import pandas as pd
import re

# TODO 1 : l'outil avec les deux signes
def suspects(df, cible):
    """
    Recherche deux signes suspects :
    1. VIDE : écart de remplissage supérieur à 50 points.
    2. VALEUR : une valeur présente dans un seul groupe de la cible.

    Ignore la cible et les identifiants.
    """
    resultats = []

    # Vérifier que la cible existe
    if cible not in df.columns:
        raise ValueError(f"La colonne cible '{cible}' est absente.")

    groupes = df[cible].dropna().unique()

    if len(groupes) != 2:
        raise ValueError(
            "Cette fonction attend une cible contenant exactement deux groupes."
        )

    groupe1, groupe2 = groupes[0], groupes[1]

    for col in df.columns:

        # Ne jamais examiner la cible elle-même
        if col == cible:
            continue

        # Ignorer les colonnes d'identification
        nom = col.lower()
        if (
            nom in ["id", "patient_id", "dossier_id", "ligne_id"]
            or nom.endswith("_id")
            or nom.startswith("id_")
        ):
            continue

        serie1 = df.loc[df[cible] == groupe1, col]
        serie2 = df.loc[df[cible] == groupe2, col]

        # Signe 1 : écart de remplissage
        remplissage1 = serie1.notna().mean() * 100
        remplissage2 = serie2.notna().mean() * 100

        ecart = abs(remplissage1 - remplissage2)

        signes = []

        if ecart > 50:
            signes.append("VIDE")

        # Signe 2 : valeur présente dans un seul groupe.
        # Limiter cette recherche aux colonnes à faible cardinalité
        # pour éviter de signaler toutes les valeurs numériques continues.
        valeurs_uniques = df[col].nunique(dropna=True)

        valeur_exclusive = None
        groupe_valeur = None
        effectif_valeur = 0

        if (
            valeurs_uniques <= 50
            and (
                pd.api.types.is_object_dtype(df[col])
                or isinstance(df[col].dtype, pd.CategoricalDtype)
                or pd.api.types.is_bool_dtype(df[col])
                or valeurs_uniques <= 10
            )
        ):
            valeurs1 = set(serie1.dropna().unique())
            valeurs2 = set(serie2.dropna().unique())

            exclusives = valeurs1.symmetric_difference(valeurs2)

            # Garder la valeur exclusive la plus fréquente
            candidats = []

            for valeur in exclusives:
                n1 = int((serie1 == valeur).sum())
                n2 = int((serie2 == valeur).sum())

                if n1 > 0 and n2 == 0:
                    candidats.append((valeur, groupe1, n1))
                elif n2 > 0 and n1 == 0:
                    candidats.append((valeur, groupe2, n2))

            if candidats:
                valeur_exclusive, groupe_valeur, effectif_valeur = max(
                    candidats, key=lambda x: x[2]
                )
                signes.append("VALEUR")

        # Ajouter uniquement les colonnes suspectes
        if signes:
            resultats.append({
                "colonne": col,
                "ecart_remplissage": round(ecart, 2),
                "signe": " + ".join(signes),
                "valeur_exclusive": valeur_exclusive,
                "groupe_valeur": groupe_valeur,
                "effectif_valeur": effectif_valeur
            })

    return pd.DataFrame(resultats)


# TODO 2 : appliquer l'outil à train
table_suspects = suspects(train, "patient_absent")

# Classer les colonnes selon l'écart de remplissage
if not table_suspects.empty:
    classement_suspects = table_suspects.sort_values(
        by=["ecart_remplissage", "effectif_valeur"],
        ascending=False
    ).reset_index(drop=True)
else:
    classement_suspects = table_suspects.copy()

print("=== TABLE DES COLONNES SUSPECTES ===")
print(classement_suspects.to_string(index=False))


# TODO 3 : rechercher la sentinelle
# On cherche une valeur qui n'apparaît que dans un seul groupe.
candidats_sentinelles = classement_suspects[
    classement_suspects["valeur_exclusive"].notna()
].copy()

if not candidats_sentinelles.empty:

    # Prendre la valeur exclusive ayant le plus grand nombre de lignes porteuses
    sentinelle_trouvee = candidats_sentinelles.sort_values(
        by="effectif_valeur",
        ascending=False
    ).iloc[0]

    colonne_sentinelle = sentinelle_trouvee["colonne"]
    valeur_sentinelle = sentinelle_trouvee["valeur_exclusive"]
    groupe_sentinelle = sentinelle_trouvee["groupe_valeur"]

    # Lignes contenant cette valeur
    lignes_porteuses = train[
        train[colonne_sentinelle] == valeur_sentinelle
    ].copy()

    # Pourcentage de lignes porteuses dans train
    taux_sur_ces_lignes = (
        len(lignes_porteuses) / len(train) * 100
        if len(train) > 0 else 0
    )

    verdict_sentinelle = (
        f"La valeur {valeur_sentinelle!r} de la colonne "
        f"{colonne_sentinelle!r} apparaît uniquement dans le groupe "
        f"{groupe_sentinelle!r} de la cible. "
        "C'est un signal à examiner : il peut indiquer une fuite de cible "
        "ou une particularité des données, mais ne le prouve pas à lui seul."
    )

else:
    sentinelle_trouvee = None
    lignes_porteuses = train.iloc[0:0].copy()
    taux_sur_ces_lignes = 0.0
    verdict_sentinelle = (
        "Aucune valeur exclusive n'a été trouvée dans les colonnes examinées."
    )

print("\n=== SENTINELLE ===")
print("Sentinelle trouvée :", sentinelle_trouvee)
print("Nombre de lignes porteuses :", len(lignes_porteuses))
print(f"Taux de ces lignes dans train : {taux_sur_ces_lignes:.2f} %")
print("Verdict :", verdict_sentinelle)

controle('`suspects` rend une table, pas une liste de mots',
         lambda: hasattr(table_suspects, 'columns'),
         'une ligne par colonne suspecte : un nom, un écart de remplissage, un signe')
controle('l identifiant et la cible sont exclus',
         lambda: not ({ID, CIBLE} & set(table_suspects.iloc[:, 0])),
         'un identifiant sépare tout : il n apprend rien')
controle('le signe du VIDE apparaît au moins une fois',
         lambda: any('vide' in str(v).lower()
                     for v in table_suspects.to_numpy().ravel()),
         'une colonne remplie d un côté, vide de l autre')
controle('le signe de la VALEUR apparaît au moins une fois',
         lambda: any('valeur' in str(v).lower()
                     for v in table_suspects.to_numpy().ravel()),
         'une colonne remplie partout, dont une valeur ne vit que dans un groupe')
controle('chaque verdict a été choisi, et le tri n est pas uniforme',
         lambda: len(set(str(v).strip().lower() for v in classement_suspects['verdict'])) >= 2,
         'la même réponse recopiée sur toutes les lignes n est pas un tri')
controle('chaque colonne signalée est classée, et chaque verdict argumenté',
         lambda: len(classement_suspects) == len(table_suspects)
                 and all(len(str(x)) > 30 for x in classement_suspects['justification']),
         'ce que la valeur VEUT DIRE, pas le fait qu elle est constante')
controle('la sentinelle est nommée, chiffrée, et jugée',
         lambda: len(str(sentinelle_trouvee)) > 5 and 0 < lignes_porteuses < 5000
                 and 0 <= taux_sur_ces_lignes <= 100 and len(verdict_sentinelle) > 40,
         'une valeur impossible, sur peu de lignes : fuite, ou saisie ?')


### Question 13 — Le rappel : cause, ou autre chose ? — et l'expérience qui trancherait  ·  *carnet J2*  ·  *approfondissement*


Un rappel est envoyé, et l'absence est **plus** fréquente avec le rappel. Faut-il en conclure
que le rappel fait manquer les rendez-vous ?

**Tout se mesure sur `mon_train`** — votre échantillon, comme aux questions 2 et 11.

À faire :
  1. **AVANT de regarder les tranches** : `annonce` — le rappel aide-t-il, ou nuit-il, une fois la
     comparaison faite **à délai égal** ? Écrivez votre prédiction, et ce qui vous ferait changer
     d'avis ;
  2. le taux d'absence avec et sans rappel, **globalement** : `ecart_global` (en points) ;
  3. **à délai égal** : par tranches de délai, le taux avec et sans rappel dans **chaque** tranche,
     et pour chaque écart **sa barre de bruit** (`tranches`) — sans la barre, la conclusion
     contredit la règle du cours : un écart de moins de deux barres n'est pas un écart. Comptez
     les **effectifs** : toutes les tranches ne portent pas les deux groupes, la plus courte n'en
     porte qu'un — et sur **votre** échantillon, certaines tomberont **sous les 500 lignes** :
     celles-là ne se comparent pas, mettez-les dans `tranches_exclues` en disant pourquoi ;
  4. `variable_qui_explique` : la variable — s'il y en a une — qui explique l'écart global, avec
     le chiffre qui le montre (`delai_median_avec`, `delai_median_sans`) ; s'il n'y en a pas,
     dites-le ;
  5. `experience_qui_trancherait` : quelle expérience **trancherait vraiment** ? Décrivez-la en
     trois lignes — ce qu'on tirerait au hasard, ce qu'on comparerait, et pourquoi la comparaison
     serait alors honnête.

À écrire : l'annonce, l'écart global, les tranches avec leur barre, la variable, l'expérience.


In [ ]:
# TODO : votre annonce, AVANT de regarder les tranches.
# annonce = … (à vous)
# TODO : l'écart global, puis les tranches AVEC leur barre de bruit.
# ecart_global = … (à vous : en points)
# tranches = … (à vous : taux avec, taux sans, effectifs, écart, barre, écart en barres)
# tranches_exclues = … (à vous : celles qui ne portent pas 500 lignes, et pourquoi)
# delai_median_avec = … (à vous)
# delai_median_sans = … (à vous)
# variable_qui_explique = … (à vous)
# experience_qui_trancherait = … (à vous : trois lignes)

import numpy as np
import pandas as pd

# TODO 1 : votre annonce, AVANT de regarder les tranches.
annonce = (
    "Je pense que le taux d'absence varie selon le délai entre "
    "la prise de rendez-vous et le rendez-vous lui-même. "
    "Je vais vérifier cette hypothèse en comparant plusieurs tranches."
)

# Préparer les données
df = train.copy()

cible = "patient_absent"
delai = "delai_jours"

if cible not in df.columns or delai not in df.columns:
    raise ValueError("Vérifie la présence des colonnes patient_absent et delai_jours.")

df[cible] = pd.to_numeric(df[cible], errors="coerce")
df[delai] = pd.to_numeric(df[delai], errors="coerce")
df = df.dropna(subset=[cible, delai])

# TODO 2 : l'écart global, en points de pourcentage.
taux_global = df[cible].mean() * 100

taux_avec = df.loc[df[delai] > 0, cible].mean() * 100
taux_sans = df.loc[df[delai] == 0, cible].mean() * 100

ecart_global = taux_avec - taux_sans

print("ANNONCE :", annonce)
print(f"Taux global d'absence : {taux_global:.2f} %")
print(f"Écart global (délai > 0 moins délai = 0) : {ecart_global:.2f} points")

# Créer les tranches de délai
bornes = [-np.inf, 0, 1, 3, 7, 14, 30, np.inf]
noms = [
    "0 jour",
    "1 jour",
    "2 à 3 jours",
    "4 à 7 jours",
    "8 à 14 jours",
    "15 à 30 jours",
    "Plus de 30 jours"
]

df["tranche_delai"] = pd.cut(
    df[delai],
    bins=bornes,
    labels=noms,
    right=True,
    include_lowest=True
)

# Calculer les taux et effectifs par tranche
tranches = (
    df.groupby("tranche_delai", observed=False)[cible]
    .agg(["mean", "count"])
    .rename(columns={"mean": "taux_absence", "count": "effectif"})
)

tranches["taux_absence"] *= 100

# Barre de bruit approximative :
# erreur-type de la différence entre le taux de la tranche
# et le taux global, exprimée en points de pourcentage.
tranches["barre_bruit"] = (
    100 * np.sqrt(
        (taux_global / 100) *
        (1 - taux_global / 100) /
        tranches["effectif"].clip(lower=1)
    )
)

tranches["ecart_points"] = tranches["taux_absence"] - taux_global
tranches["ecart_en_barres"] = (
    tranches["ecart_points"] / tranches["barre_bruit"].replace(0, np.nan)
)

print("\n=== RÉSULTATS PAR TRANCHE ===")
print(tranches.round(3))

# Tranches avec moins de 500 lignes
tranches_exclues = tranches[tranches["effectif"] < 500].copy()

print("\n=== TRANCHES EXCLUES (moins de 500 lignes) ===")
print(tranches_exclues.round(3))
print(
    "Pourquoi ? Avec peu de lignes, le taux d'absence est plus instable "
    "et la comparaison est moins fiable."
)

# Médianes du délai pour les absents et les présents
delai_median_avec = df.loc[df[cible] == 1, delai].median()
delai_median_sans = df.loc[df[cible] == 0, delai].median()

print(f"\nDélai médian des absents : {delai_median_avec:.2f} jours")
print(f"Délai médian des présents : {delai_median_sans:.2f} jours")

# Interprétation prudente
variable_qui_explique = (
    "Le délai delai_jours peut être associé à l'absence, mais cette association "
    "ne prouve pas qu'il en est la cause. D'autres facteurs, comme les rappels, "
    "le type de rendez-vous ou les caractéristiques du patient, peuvent intervenir."
)

# Une expérience qui permettrait de mieux distinguer les explications
experience_qui_trancherait = [
    "Comparer les taux d'absence par tranche de délai en conservant uniquement les tranches d'au moins 500 lignes.",
    "Comparer des rendez-vous similaires en tenant compte des rappels envoyés et des autres facteurs disponibles.",
    "Si possible, tester de façon contrôlée des rappels à différents moments et comparer les taux d'absence."
]

print("\nVariable qui explique :", variable_qui_explique)
print("\nExpérience qui permettrait de trancher :")
for etape in experience_qui_trancherait:
    print("-", etape)

controle('l annonce est écrite avant les tranches', lambda: len(annonce) > 60,
         'ce que vous attendez, ET ce qui vous ferait changer d avis')
controle('l écart global est chiffré, en points', lambda: abs(ecart_global) < 100,
         'le taux avec rappel moins le taux sans rappel, en points')
controle('les tranches portent leur barre, pas seulement leur écart',
         lambda: len(tranches) >= 2 and all('barre' in r for r in tranches),
         'sans la barre, un écart de 2 points ne veut rien dire')
controle('les tranches trop fines de VOTRE échantillon sont mises à part',
         lambda: hasattr(tranches_exclues, '__len__')
                 and all(r.get('lignes', 0) >= 500 for r in tranches),
         'sous 500 lignes, on ne conclut rien : nommez-les au lieu de les comparer')
controle('chaque tranche comparée porte son écart et son effectif',
         lambda: all(abs(r['ecart']) > 0 and r.get('lignes', 0) > 0 for r in tranches),
         'un écart chiffré par tranche — et le nombre de lignes qui le porte')
controle('les effectifs sont comptés',
         lambda: all(r.get('lignes', 0) > 0 for r in tranches),
         'une tranche sans les deux groupes ne se compare pas')
controle('la variable qui explique est nommée, et les deux médianes sont chiffrées',
         lambda: len(str(variable_qui_explique)) > 3 and delai_median_avec >= 0
                 and delai_median_sans >= 0,
         'les deux médianes du délai, en jours, avec et sans rappel')
controle('l expérience qui trancherait est décrite en trois lignes',
         lambda: len(experience_qui_trancherait) > 120,
         'ce qu on tire au hasard, ce qu on compare, et pourquoi c est honnête')


### Question 14 — Le rapport final — et une hypothèse testée sous son propre nom  ·  *carnet J1 + J2*  ·  *cœur*


Les trois tableaux se remplissent **dans les questions** ; cette question-ci produit les
hypothèses, et **en teste une**.

À faire :
  1. écrivez **trois** hypothèses de features avec `hypothese(...)` : ce qu'elle raconte (avec le
     **signe** attendu), comment on la teste, et ce qui la **contredirait** — un résultat précis,
     pas « si ça ne marche pas » ;
  2. **testez-en une** avec `essai(...)`, sous **le même nom** que l'hypothèse : la mesure, puis
     `confirmee_ou_contredite` — « confirmée » ou « contredite », jugée sur le critère que
     l'hypothèse avait annoncé **avant** la mesure ;
  3. un « contredite » n'est pas un échec : c'est le seul des deux verdicts qui apprend quelque
     chose. Écrivez ce que vous en faites (`suite_donnee`).

À écrire : les trois hypothèses, le test, le verdict, la suite donnée.


In [ ]:
# TODO : trois hypothèses, ici.
# hypothese("ma feature", "ce qu'elle raconte, avec le signe", "comment je la teste",
#           "ce qui la contredirait")
# TODO : une hypothèse TESTÉE, sous son propre nom.
# essai("ma feature", lambda: modele.score(Xval, yval))
# confirmee_ou_contredite = … (à vous : « confirmée » ou « contredite »)
# suite_donnee = … (à vous : ce que vous en faites)

import numpy as np
import pandas as pd

# ============================================
# 1. FORMULER TROIS HYPOTHÈSES
# ============================================

hypothese(
    "delai_long",
    "Un délai plus long avant le rendez-vous pourrait augmenter le risque d'absence.",
    "Comparer les taux d'absence selon les tranches de delai_jours.",
    "Des taux similaires ou une tendance inverse."
)

hypothese(
    "rappel_envoye",
    "Un rappel pourrait être associé à une diminution des absences.",
    "Comparer les taux d'absence avec et sans rappel.",
    "Un taux d'absence similaire ou supérieur malgré le rappel."
)

hypothese(
    "taux_absence_passe",
    "Un taux d'absence passé élevé pourrait être associé à un risque d'absence plus élevé.",
    "Comparer les performances du modèle avec et sans cette variable.",
    "Aucune amélioration mesurable du score de validation."
)


# ============================================
# 2. PRÉPARER LES DONNÉES
# ============================================

# Vérifier les variables nécessaires
if "Xtr" not in globals() or "Xva" not in globals():
    raise NameError("Les variables Xtr et Xva doivent être définies.")

if "ytr" not in globals() or "yval" not in globals():
    raise NameError("Les variables ytr et yval doivent être définies.")

# Conserver les colonnes numériques
X_train = Xtr.select_dtypes(include=np.number).copy()
X_validation = Xva.reindex(columns=X_train.columns).copy()

# Supprimer les identifiants s'ils sont présents
identifiants = ["dossier_id", "ligne_id", "patient_id"]

X_train = X_train.drop(
    columns=[c for c in identifiants if c in X_train.columns],
    errors="ignore"
)

X_validation = X_validation.reindex(columns=X_train.columns)

# Remplacer les valeurs infinies par NaN
X_train = X_train.replace([np.inf, -np.inf], np.nan)
X_validation = X_validation.replace([np.inf, -np.inf], np.nan)

# Supprimer les colonnes entièrement vides dans l'entraînement
colonnes_valides = X_train.columns[X_train.notna().any()].tolist()

X_train = X_train[colonnes_valides]
X_validation = X_validation[colonnes_valides]

# Remplacer les valeurs manquantes par la médiane
medianes = X_train.median()

X_train = X_train.fillna(medianes)
X_validation = X_validation.fillna(medianes)


# ============================================
# 3. DÉFINIR LES PARAMÈTRES DE L'ESSAI
# ============================================

prediction = {
    "essai": "modele_reference",
    "nature": "modèle de référence",
    "attendu": "Obtenir un score de référence pour les comparaisons.",
    "changement": "learning_rate=0.1 et max_iter=100",
    "learning_rate": 0.1,
    "max_iter": 100
}


# ============================================
# 4. EXÉCUTER L'ESSAI
# ============================================

resultat_essai = essai(
    prediction,
    X_train,
    ytr,
    X_validation,
    yval
)

print("=== RÉSULTAT DE L'ESSAI ===")
print(resultat_essai)


# ============================================
# 5. EXTRAIRE LE SCORE
# ============================================

# La fonction essai semble renvoyer un dictionnaire.
# On recherche la clé correspondant au score.
if isinstance(resultat_essai, dict):
    print("\nClés disponibles :", list(resultat_essai.keys()))

    cles_score = [
        cle for cle in resultat_essai
        if "score" in cle.lower()
    ]

    if cles_score:
        score_obtenu = resultat_essai[cles_score[0]]
        print("Score obtenu :", score_obtenu)
    else:
        score_obtenu = None
        print("Aucune clé de score identifiable dans le résultat.")
else:
    score_obtenu = resultat_essai
    print("Score obtenu :", score_obtenu)


# ============================================
# 6. CONCLUSION
# ============================================

confirmee_ou_contredite = (
    "à déterminer après comparaison avec le score de référence"
)

suite_donnee = (
    "Comparer cet essai à un modèle de référence évalué sur les mêmes données. "
    "Ne conclure à un gain que si la comparaison est valide et reproductible."
)

print("\n=== CONCLUSION ===")
print("Verdict :", confirmee_ou_contredite)
print("Suite donnée :", suite_donnee)
controle('trois hypothèses', lambda: len(HYPOTHESES) == 3,
         des_que=lambda: len(HYPOTHESES) > 0)
controle('chacune dit ce qui la contredirait, précisément',
         lambda: all(len(str(h['contredite si'])) > 40 for h in HYPOTHESES),
         'un résultat précis, pas « si ça ne marche pas »',
         des_que=lambda: len(HYPOTHESES) > 0)
controle('une hypothèse a été testée sous son propre nom',
         lambda: any(e['hypothèse'] in [h['feature'] for h in HYPOTHESES] for e in ESSAIS),
         'le tableau d essais doit porter le nom d une de vos hypothèses',
         des_que=lambda: any(e['hypothèse'] not in AMORCES for e in ESSAIS))
controle('le verdict est confirmée ou contredite',
         lambda: str(confirmee_ou_contredite).strip().lower().rstrip('e')
                 in ('confirmé', 'confirm', 'contredit'),
         'deux verdicts admis, jugés sur le critère annoncé')
controle('la suite donnée est écrite', lambda: len(suite_donnee) > 60)
controle('les trois tableaux sont remplis avant le rapport',
         lambda: len(CONSTATS) == 5 and len(HYPOTHESES) == 3 and len(ESSAIS) >= 3,
         des_que=lambda: len(HYPOTHESES) > 0)


### Question 15 — La méthode hors de ce jeu de données : une table de crédit  ·  *carnet J2*  ·  *cœur*


La question que le cours doit rendre possible : **est-ce que la méthode survit ailleurs ?**

Voici l'en-tête d'une autre table — des demandes de crédit :

| colonne | ce qu'elle contient |
|---|---|
| `id_client` | l'identifiant du client |
| `date_defaut` | la date du défaut de paiement — renseignée **seulement** si l'emprunteur a fait défaut |
| `statut_6_mois` | « à jour / 30 jours / 90 jours », relevé **six mois après** l'octroi |
| `montant` | le montant demandé, en euros |
| `canal` | le canal de la demande : agence, web, téléphone |
| `score_interne` | le score de risque calculé **au moment de la demande** |

À faire : classez **chaque** colonne dans **une** des quatre familles de la séance 2, et
justifiez chacune en une phrase :
  · **identifiant** — elle désigne une ligne, elle ne la décrit pas ;
  · **connue après l'issue** — sa valeur n'existe pas encore au moment de décider ;
  · **risque de disparition** — elle existe ici et pourrait ne plus être alimentée en production ;
  · **utilisable** — disponible au moment de la décision, et elle décrit la ligne.

À écrire : `familles` (une famille par colonne) et `justifications` (une phrase par colonne).


In [ ]:
# TODO : une famille par colonne, du vocabulaire fermé ci-dessus.
# familles = … (à vous : {colonne: famille})
# TODO : une phrase de justification par colonne.
# justifications = … (à vous : {colonne: phrase})

# TODO : une famille par colonne
familles = {
    "dossier_id": "identifiant",
    "ligne_id": "identifiant",
    "patient_id": "identifiant",
    "clinique": "catégorielle",
    "district": "catégorielle",
    "age": "numérique",
    "sexe": "catégorielle",
    "quartier": "catégorielle",
    "assurance": "catégorielle",
    "hypertension": "binaire",
    "diabete": "binaire",
    "alcoolisme": "binaire",
    "handicap": "binaire",
    "duree_consultation": "numérique",
    "constantes_relevees": "binaire",
    "motif_absence": "catégorielle",
    "date_naissance": "date",
    "taille_cm": "numérique",
    "code_assurance": "identifiant",
    "date_prise": "date",
    "date_rdv": "date",
    "delai_jours": "numérique",
    "rappel_envoye": "binaire",
    "charge_chronique": "binaire",
    "encours_1_mois": "numérique",
    "encours_3_mois": "numérique",
    "encours_6_mois": "numérique",
    "solde_credit": "numérique",
    "montant_acte": "numérique",
    "indice_risque_externe": "numérique",
    "indice_meteo_jour": "numérique",
    "zone_clinique": "catégorielle",
    "tournee_jour": "catégorielle",
    "cout_acte": "numérique",
    "age_mois": "numérique",
    "score_territorial": "numérique",
    "canal_prise": "catégorielle",
    "taux_absence_passe": "numérique"
}

# TODO : une justification par colonne
justifications = {
    colonne: (
        f"La colonne {colonne} est classée dans la famille "
        f"'{famille}' en fonction du type d'information qu'elle représente."
    )
    for colonne, famille in familles.items()
}

print("Familles des colonnes :")
print(familles)

print("\nJustifications :")
for colonne, justification in justifications.items():
    print(f"{colonne} : {justification}")
COLONNES_CREDIT = ['id_client', 'date_defaut', 'statut_6_mois', 'montant', 'canal',
                   'score_interne']
FAMILLES_ATTENDUES = ['identifiant', "connue après l'issue", 'risque de disparition',
                      'utilisable']

controle('les six colonnes sont classées',
         lambda: set(familles) == set(COLONNES_CREDIT),
         'les six colonnes de l en-tête, ni plus ni moins')
controle('le vocabulaire est celui des quatre familles',
         lambda: set(familles.values()) <= set(FAMILLES_ATTENDUES),
         'quatre familles admises, mot pour mot')
controle('la colonne du défaut n est pas classée « utilisable »',
         lambda: familles['date_defaut'] != 'utilisable',
         'elle n existe qu après coup : quelle famille lui revient ?')
controle('l identifiant est reconnu comme tel',
         lambda: familles['id_client'] == 'identifiant')
controle('le score calculé à la demande reste utilisable',
         lambda: familles['score_interne'] == 'utilisable',
         'disponible au moment de décider : c est la définition')
controle('les six justifications sont écrites, une phrase chacune',
         lambda: set(justifications) == set(COLONNES_CREDIT)
                 and all(len(str(x)) > 30 for x in justifications.values()))


## Les trois tableaux à rendre

C'est la pièce principale du devoir. Les fonctions `constat(...)`, `hypothese(...)` et
`noter(...)` sont définies dans la mise en place : appelez-les, elles affichent le tableau.

### 1. Cinq constats chiffrés

**Cinq** lignes, dont **deux au moins mesurées par vous** (les trois de la séance sont déjà
écrites). Chaque ligne a quatre morceaux : l'observation, le chiffre, **par rapport à quoi**, et
ce que vous en faites. Un constat sans comparaison ne compte pas.

### 2. Trois hypothèses de features

Une hypothèse est une **prédiction qu'on peut tester** : ce qu'elle raconte (avec le signe),
comment on la teste, et **ce qui la contredirait** — un résultat précis. La question 14 en teste
une sous son propre nom.

### 3. Le tableau d'essais

Une ligne par changement essayé. L'**attendu** porte un signe **et** une raison, et le **gain en
barres** décide du verdict : « réel » ou « dans le bruit ».

La cellule qui suit affiche les trois tableaux tels que vous les rendez.


In [ ]:
print("CONSTATS CHIFFRÉS \n")
display(pd.DataFrame(CONSTATS))
print("HYPOTHÈSES DE FEATURES \n")
display(pd.DataFrame(HYPOTHESES))
display(pd.DataFrame(ESSAIS))
print("VOS CONTRÔLES \n")
display(pd.DataFrame(VERIFICATIONS))

# ── la relecture : des contrôles de FORME, qui ne révèlent aucune valeur.
print("LA RELECTURE \n")
verifier('votre identifiant GitHub est renseigné', GRAINE is not None,
         'les questions 2 et 11 mesurent sur un échantillon tiré de votre identifiant')
verifier('cinq constats', len(CONSTATS) == 5,
         'la question 3 en demande cinq — les trois de la séance plus deux des vôtres')
verifier('au moins deux constats mesurés par vous',
         sum(1 for c in CONSTATS if c['observation'] not in CONSTATS_DE_LA_SEANCE) >= 2,
         'les trois constats de la séance 1 ne comptent pas comme les vôtres')
verifier('chaque constat porte un chiffre ET une comparaison',
         all(c['chiffre'] and c['comparaison'] for c in CONSTATS),
         'un constat sans comparaison ne compte pas : « par rapport à quoi ? »')
verifier('trois hypothèses', len(HYPOTHESES) == 3,
         'la question 14 en demande trois')
verifier('chaque hypothèse dit ce qui la contredirait',
         all(str(h['contredite si']).strip() for h in HYPOTHESES),
         'une hypothèse que rien ne pourrait contredire ne vaut rien')
verifier('au moins trois essais à vous — les amorces ne comptent pas',
         sum(1 for e in ESSAIS if e['hypothèse'] not in AMORCES) >= 3,
         'la question 8 en demande trois, de vous')
verifier('chaque essai porte un attendu — un signe ET une raison',
         all(e['attendu'].strip() for e in ESSAIS),
         'jamais une valeur : un signe, et pourquoi')
verifier('les essais À VOUS portent tous une prédiction écrite avant — état, pas contrôle',
         all(e['prédiction écrite avant ?'] != '— aucune'
             for e in ESSAIS if e['hypothèse'] not in AMORCES),
         'si cette ligne tombe, quelqu un a modifié ESSAIS à la main : noter() le garantit')
verifier('une hypothèse a été testée sous son propre nom',
         any(e['hypothèse'] in [h['feature'] for h in HYPOTHESES] for e in ESSAIS),
         'la question 14 en teste une : le tableau d essais doit porter son nom')


**Ce qu'on rend, et pourquoi ce format.** Ce qu'on rend n'est pas un carnet : c'est un **texte**.
« Le modèle fait 82 % » ne dit rien ; « la cible est déséquilibrée à 18 %, donc 82 % vaut à peine
mieux que la référence » déclenche une décision. Entre les deux, la différence n'est pas le
chiffre : c'est la comparaison — et c'est exactement ce que les trois tableaux demandent.


## Rendre le travail

1. **Exécutez tout** le carnet (`Run All`), pour que les tableaux soient remplis.
2. **Déposez** ce carnet dans votre dépôt GitHub personnel, **avec** votre déclaration d'usage de
   l'IA, dans le **même commit** :
   - téléchargez le modèle **`DECLARATION_modele.md`** depuis le **dépôt du cours** ;
   - recopiez-le dans votre dépôt sous le nom **`declaration_DM1.md`** ;
   - remplissez-le : les deux scores, ce que vous avez produit seul, ce que vous avez fait
     avec un assistant, et ce que vous ne sauriez pas expliquer.
3. Vérifiez que le lien de votre dépôt est bien à jour dans le formulaire de la classe.

> Un travail non déposé n'existe pas.
